# Advanced Retriever Lab — Start Here

Use this notebook as a **diagnose → choose → run** workbook for RAG retrieval techniques.

### How to use the table below
1. **Find the row** that matches the problem you are seeing (missed IDs, vague chat follow-ups, repetitive chunks, etc.).
2. **Try that technique first** — it is usually the highest-leverage fix for that failure mode.
3. **Jump to the matching practical section** in this notebook and compare it against a plain dense baseline.
4. **Ask one question after each run:** *Did this fix the failure mode, or only change the ranking noise?*

> The table is a **use-case chooser**, not a full curriculum. Read left-to-right: symptom → technique → why it fits.


| Problem you are seeing                                     | Best technique to try first   | Why                                              |
| ---------------------------------------------------------- | ----------------------------- | ------------------------------------------------ |
| Exact names, IDs, codes, dates are missed                  | **BM25 / Sparse**             | Exact lexical matching                           |
| User uses different words than documents                   | **Dense Retrieval**           | Semantic matching                                |
| Both exact terms + semantic intent matter                  | **Hybrid Retrieval**          | BM25 + dense complement each other               |
| BM25/dense result lists need merging                       | **RRF**                       | Rank-based fusion avoids score-scale problems    |
| You know one retriever should matter more                  | **Weighted Fusion**           | Explicitly control contribution                  |
| Retrieved chunks are repetitive                            | **MMR**                       | Relevance + diversity                            |
| Conversational query is incomplete                         | **Query Rewriting**           | Makes query standalone                           |
| Different wording causes poor recall                       | **Multi-Query**               | Searches multiple formulations                   |
| Very short/abstract query does not match document language | **HyDE**                      | Converts query into document-like representation |
| One question contains multiple independent questions       | **Query Decomposition**       | Retrieve each sub-question separately            |
| Next search depends on information found in first search   | **Multi-Hop Retrieval**       | Sequential dependent retrieval                   |
| Small chunks retrieve well but lack context                | **Parent Document Retrieval** | Search small → return larger parent              |
| Exact sentence retrieves well but needs nearby context     | **Sentence Window**           | Search sentence → expand locally                 |
| Correct docs retrieved but wrong one ranks first           | **Reranking**                 | More accurate second-stage ranking               |
| Correct docs contain lots of irrelevant text               | **Contextual Compression**    | Remove irrelevant context                        |
| Tenant/role/year/version restrictions exist                | **Metadata Pre-filtering**    | Restrict search before retrieval                 |
| Data exists in SQL + vector DB + web + APIs                | **Query Routing**             | Select correct source                            |
| Answer exists in image/table/diagram                       | **Multimodal retrieval**      | Text retrieval alone is insufficient             |


### Learning: RETRIEVER + HYDE PRACTICAL

**What you'll learn:** Embed a guessed answer, then search near that vector.

**What this cell does:** Runs `RETRIEVER + HYDE PRACTICAL` and prints intermediate results you can inspect.

**Watch for:** Helps when the question vocabulary differs from the docs.



In [ ]:
# ============================================================
# RETRIEVER + HYDE PRACTICAL
# ============================================================

### Learning: !pip install langchain_community langchain_text_splitters langchain_op

**What you'll learn:** Install the packages this notebook needs.

**What this cell does:** Installs required Python packages into the runtime.

**Watch for:** Run once; restart runtime if Colab asks.



In [ ]:
!pip install langchain_community langchain_text_splitters langchain_openai langchain_chroma pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 2.2 MB/s eta 0:00:00


### Learning: IMPORTS

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** Runs `IMPORTS` and prints intermediate results you can inspect.

**Watch for:** If an import fails, re-run the install cell.



In [ ]:
print("All imports and setup starting...")

# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
import getpass
import os
import shutil

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

from langchain_classic.chains.hyde.base import (
    HypotheticalDocumentEmbedder
)

All imports and setup starting...


/tmp/ipykernel_520/2286258816.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### Learning: from google.colab import userdata

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** Runs `from google.colab import userdata` and prints intermediate results you can inspect.

**Watch for:** If an import fails, re-run the install cell.



In [ ]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

### Learning: OPENAI API KEY

**What you'll learn:** Authenticate so embedding and chat calls can run.

**What this cell does:** Runs `OPENAI API KEY` and prints intermediate results you can inspect.

**Watch for:** Never hardcode secrets in shared notebooks.



In [ ]:
# ============================================================
# 2. OPENAI API KEY
# ============================================================

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass(
        "Enter your OpenAI API key: "
    )

print("OpenAI API key configured successfully.")

OpenAI API key configured successfully.


### Learning: DATA DIRECTORY

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** Runs `DATA DIRECTORY` and prints intermediate results you can inspect.

**Watch for:** Confirm page count and first-page text look sane.



In [ ]:
# ============================================================
# 3. DATA DIRECTORY
# ============================================================

DATA_DIR = Path(
    r"/content/"
)

preferred_pdf = DATA_DIR / "llama2-research-paper.pdf"

### Learning: FIND PDF

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** Runs `FIND PDF` and prints intermediate results you can inspect.

**Watch for:** Confirm page count and first-page text look sane.



In [ ]:
# ============================================================
# 4. FIND PDF
# ============================================================

if preferred_pdf.exists():

    PDF_PATH = preferred_pdf

else:

    available_pdfs = sorted(
        DATA_DIR.glob("*.pdf")
    )

    if len(available_pdfs) == 1:

        PDF_PATH = available_pdfs[0]

    elif len(available_pdfs) == 0:

        raise FileNotFoundError(
            f"No PDF file was found inside:\n{DATA_DIR}"
        )

    else:

        raise RuntimeError(
            "Multiple PDF files were found. "
            "Please set PDF_PATH manually.\n"
            + "\n".join(
                str(path)
                for path in available_pdfs
            )
        )


print("PDF found:")
print(PDF_PATH)

PDF found:
/content/llama2-research-paper.pdf


### Learning: LOAD PDF

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** Runs `LOAD PDF` and prints intermediate results you can inspect.

**Watch for:** Confirm page count and first-page text look sane.



In [15]:
# ============================================================
# 5. LOAD PDF
# ============================================================

loader = PyPDFLoader(
    str(PDF_PATH)
)

pages = loader.load()

print(
    f"\nTotal PDF pages loaded: {len(pages)}"
)


# ============================================================
# 6. INSPECT FIRST PAGE
# ============================================================

print("\nFirst-page metadata:")
print(
    pages[0].metadata
)

print("\nFirst 1,000 characters:")
print(
    pages[0].page_content[:1000]
)


Total PDF pages loaded: 77

First-page metadata:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/content/llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1'}

First 1,000 characters:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Art

### Learning: IDENTIFY PAPER SECTIONS

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Defines helper logic for: IDENTIFY PAPER SECTIONS.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 7. IDENTIFY PAPER SECTIONS
# ============================================================

def identify_section(
    paper_page: int
) -> str:

    if 1 <= paper_page <= 2:
        return "front_matter"

    if 3 <= paper_page <= 4:
        return "introduction"

    if 5 <= paper_page <= 7:
        return "pretraining"

    if 8 <= paper_page <= 19:
        return "fine_tuning"

    if 20 <= paper_page <= 31:
        return "safety"

    if 32 <= paper_page <= 35:
        return "discussion"

    if paper_page == 36:
        return "conclusion"

    if 37 <= paper_page <= 45:
        return "references"

    if 46 <= paper_page <= 77:
        return "appendix"

    return "unknown"


# ============================================================
# 8. ADD METADATA
# ============================================================

for page_document in pages:

    page_index = int(
        page_document.metadata.get(
            "page",
            0
        )
    )

    paper_page = (
        page_index + 1
    )

    page_document.metadata.update(
        {
            "paper": "Llama 2",
            "organization": "Meta",
            "year": 2023,
            "document_type": "research_paper",
            "paper_page": paper_page,
            "section": identify_section(
                paper_page
            ),
            "access_level": "public",
        }
    )


print("\nMetadata after enrichment:")

for page_document in pages[:5]:

    print(
        page_document.metadata
    )



Metadata after enrichment:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\complete_content_new\\Full-Stack-GenAI-Bootcamp-1.0\\Class-37-08-Aug-2026-prompting\\data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1', 'paper': 'Llama 2', 'organization': 'Meta', 'year': 2023, 'document_type': 'research_paper', 'paper_page': 1, 'section': 'front_matter', 'access_level': 'public'}
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',

### Learning: TEXT SPLITTING

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `TEXT SPLITTING` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 9. TEXT SPLITTING
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

chunks = text_splitter.split_documents(
    pages
)

print(
    f"\nTotal pages: {len(pages)}"
)

print(
    f"Total chunks: {len(chunks)}"
)


# ============================================================
# 10. ADD CHUNK IDs
# ============================================================

for chunk_number, chunk in enumerate(
    chunks
):

    paper_page = chunk.metadata.get(
        "paper_page",
        "unknown"
    )

    chunk.metadata[
        "chunk_id"
    ] = (
        f"llama2-page-"
        f"{paper_page}-"
        f"chunk-{chunk_number}"
    )


print("\nFirst chunk content:")

print(
    chunks[0].page_content[:1000]
)

print("\nFirst chunk metadata:")

print(
    chunks[0].metadata
)


Total pages: 77
Total chunks: 343

First chunk content:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich
Yinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra
Igor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi
Alan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tang
Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang
Angela Fan Melanie Kambadur Shar

### Learning: CREATE EMBEDDING MODEL

**What you'll learn:** Authenticate so embedding and chat calls can run.

**What this cell does:** Runs `CREATE EMBEDDING MODEL` and prints intermediate results you can inspect.

**Watch for:** Never hardcode secrets in shared notebooks.



In [ ]:
# ============================================================
# 11. CREATE EMBEDDING MODEL
# ============================================================

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)


# ============================================================
# 12. TEST EMBEDDING MODEL
# ============================================================

test_vector = embeddings.embed_query(
    "What is Llama 2?"
)

print(
    f"\nEmbedding dimensions: "
    f"{len(test_vector)}"
)

print(
    f"First 10 values: "
    f"{test_vector[:10]}"
)



Embedding dimensions: 1536
First 10 values: [0.0027942657470703125, -0.0521240234375, -0.021087646484375, -0.055419921875, -0.026397705078125, 0.028961181640625, -0.002071380615234375, 0.034759521484375, -0.0164794921875, -0.0245208740234375]


### Learning: CHROMA CONFIGURATION

**What you'll learn:** Build or reload the vector index used by retrievers.

**What this cell does:** Runs `CHROMA CONFIGURATION` and prints intermediate results you can inspect.

**Watch for:** Use REBUILD_INDEX=False after the first successful build.



In [ ]:
# ============================================================
# 13. CHROMA CONFIGURATION
# ============================================================

PERSIST_DIRECTORY = (
    DATA_DIR
    / "chroma_llama2_retriever"
)

COLLECTION_NAME = (
    "llama2_retriever_demo"
)


### Learning: CREATE OR LOAD VECTOR STORE

**What you'll learn:** Build or reload the vector index used by retrievers.

**What this cell does:** Runs `CREATE OR LOAD VECTOR STORE` and prints intermediate results you can inspect.

**Watch for:** Use REBUILD_INDEX=False after the first successful build.



In [ ]:
# ============================================================
# 14. CREATE OR LOAD VECTOR STORE
# ============================================================

# True  = rebuild complete vector DB
# False = reuse existing vector DB

REBUILD_INDEX = True

### Learning: VERIFY VECTOR STORE

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Runs `VERIFY VECTOR STORE` and prints intermediate results you can inspect.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
if REBUILD_INDEX:

    print(
        "\nRebuilding vector store..."
    )

    if PERSIST_DIRECTORY.exists():

        shutil.rmtree(
            PERSIST_DIRECTORY,
            ignore_errors=True
        )

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=str(
            PERSIST_DIRECTORY
        ),
        collection_configuration={
            "hnsw": {
                "space": "cosine"
            }
        },
    )

    print(
        "New vector store created."
    )

else:

    if not PERSIST_DIRECTORY.exists():

        print(
            "\nExisting vector DB "
            "not found."
        )

        print(
            "Creating a new vector store..."
        )

        vector_store = Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            collection_name=COLLECTION_NAME,
            persist_directory=str(
                PERSIST_DIRECTORY
            ),
            collection_configuration={
                "hnsw": {
                    "space": "cosine"
                }
            },
        )

        print(
            "New vector store created."
        )

    else:

        print(
            "\nLoading existing "
            "vector store..."
        )

        vector_store = Chroma(
            collection_name=COLLECTION_NAME,
            embedding_function=embeddings,
            persist_directory=str(
                PERSIST_DIRECTORY
            ),
        )

        print(
            "Existing vector store "
            "loaded."
        )


# ============================================================
# 15. VERIFY VECTOR STORE
# ============================================================

stored_count = (
    vector_store
    ._collection
    .count()
)

print(
    f"\nStored chunks: "
    f"{stored_count}"
)

print(
    f"Persisted at: "
    f"{PERSIST_DIRECTORY}"
)



Rebuilding vector store...
New vector store created.

Stored chunks: 343
Persisted at: D:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\Class-37-08-Aug-2026-prompting\data\chroma_llama2_retriever


### Learning: NORMAL DENSE RETRIEVAL TEST

**What you'll learn:** Retrieve by semantic nearest-neighbors.

**What this cell does:** Runs `NORMAL DENSE RETRIEVAL TEST` and prints intermediate results you can inspect.

**Watch for:** Dense can miss exact codes/IDs — compare with BM25 later.



In [ ]:
# ============================================================
# 16. NORMAL DENSE RETRIEVAL TEST
# ============================================================

query = (
    "How does Llama 2 improve safety?"
)

### Learning: normal_documents = (

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Executes retrieval/generation for: normal_documents = (.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
normal_documents = (
    vector_store
    .similarity_search(
        query=query,
        k=4
    )
)


print(
    "\nNORMAL DENSE RETRIEVAL"
)

print(
    "=" * 80
)

for i, document in enumerate(
    normal_documents,
    start=1
):

    print(
        f"\nResult {i}"
    )

    print(
        "Page:",
        document.metadata.get(
            "paper_page"
        )
    )

    print(
        "Section:",
        document.metadata.get(
            "section"
        )
    )

    print(
        document.page_content[:700]
    )


NORMAL DENSE RETRIEVAL

Result 1
Page: 3
Section: introduction
continue to improve the safety of those models, paving the way for more responsible development of LLMs.
We also share novel observations we made during the development ofLlama 2 and Llama 2-Chat, such as
the emergence of tool usage and temporal organization of knowledge.
3

Result 2
Page: 3
Section: introduction
the community to advance AI alignment research.
In this work, we develop and release Llama 2, a family of pretrained and fine-tuned LLMs,Llama 2 and
Llama 2-Chat, at scales up to 70B parameters. On the series of helpfulness and safety benchmarks we tested,
Llama 2-Chat models generally perform better than existing open-source models. They also appear to
be on par with some of the closed-source models, at least on the human evaluations we performed (see
Figures 1 and 3). We have taken measures to increase the safety of these models, using safety-specific data
annotation and tuning, as well as conducting red-teaming

### Learning: CREATE HYDE EMBEDDINGS

**What you'll learn:** Authenticate so embedding and chat calls can run.

**What this cell does:** Runs `CREATE HYDE EMBEDDINGS` and prints intermediate results you can inspect.

**Watch for:** Never hardcode secrets in shared notebooks.



In [ ]:
# ============================================================
# 17. CREATE HYDE EMBEDDINGS
# ============================================================

hyde_embeddings = (
    HypotheticalDocumentEmbedder
    .from_llm(
        llm=OpenAI(
            model="gpt-3.5-turbo-instruct",
            temperature=0
        ),
        base_embeddings=embeddings,
        prompt_key="web_search"
    )
)


print(
    "\nHyDE embedding model created."
)


HyDE embedding model created.


### Learning: HYDE FLOW

**What you'll learn:** Create vectors that put similar meaning nearby.

**What this cell does:** Runs `HYDE FLOW` and prints intermediate results you can inspect.

**Watch for:** Note dimension size; it must match the index.



In [ ]:
# ============================================================
# 18. HYDE FLOW
# ============================================================

"""
User Query
    ↓
LLM
    ↓
Hypothetical Document
    ↓
Embedding Model
    ↓
HyDE Query Vector
    ↓
Vector Search
    ↓
Real Documents
"""

### Learning: GENERATE HYPOTHETICAL DOCUMENT

**What you'll learn:** Create vectors that put similar meaning nearby.

**What this cell does:** Executes retrieval/generation for: GENERATE HYPOTHETICAL DOCUMENT.

**Watch for:** Note dimension size; it must match the index.



In [ ]:
# ============================================================
# 19. GENERATE HYPOTHETICAL DOCUMENT
# ============================================================

query = (
    "How does Llama 2 improve safety?"
)

hypothetical_document = (
    hyde_embeddings
    .llm_chain
    .invoke(
        {
            "QUESTION": query
        }
    )
)


print(
    "\nORIGINAL QUERY:"
)

print(
    query
)

print(
    "\nHYPOTHETICAL DOCUMENT:"
)

print(
    hypothetical_document
)



ORIGINAL QUERY:
How does Llama 2 improve safety?

HYPOTHETICAL DOCUMENT:
 Llama 2 is a revolutionary safety system that has been designed to enhance safety in various settings. This innovative system utilizes advanced technology and cutting-edge features to provide a comprehensive safety solution. One of the main ways in which Llama 2 improves safety is through its real-time monitoring capabilities. The system is equipped with sensors and cameras that constantly monitor the environment and detect any potential hazards or risks. This allows for immediate action to be taken, preventing accidents or incidents from occurring. Additionally, Llama 2 has a built-in emergency response feature that can be activated in case of an emergency. This feature quickly alerts the necessary authorities and provides them with the exact location of the incident, allowing for a swift and efficient response. Furthermore, Llama 2 also has a user-friendly interface that allows individuals to easily access saf

### Learning: GENERATE HYDE VECTOR

**What you'll learn:** Create vectors that put similar meaning nearby.

**What this cell does:** Runs `GENERATE HYDE VECTOR` and prints intermediate results you can inspect.

**Watch for:** Note dimension size; it must match the index.



In [ ]:


# ============================================================
# 20. GENERATE HYDE VECTOR
# ============================================================

hyde_vector = (
    hyde_embeddings
    .embed_query(
        query
    )
)


print(
    "\nHyDE embedding dimension:",
    len(hyde_vector)
)

print(
    "\nFirst 10 embedding values:"
)

print(
    hyde_vector[:10]
)



HyDE embedding dimension: 1536

First 10 embedding values:
[0.01241302490234375, -0.01287078857421875, 0.0023708343505859375, -0.00775146484375, -0.0204620361328125, -0.002941131591796875, 0.011077880859375, 0.0184783935546875, 0.0157470703125, 0.02001953125]


### Learning: SEARCH USING HYDE VECTOR

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Executes retrieval/generation for: SEARCH USING HYDE VECTOR.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 21. SEARCH USING HYDE VECTOR
# ============================================================

hyde_documents = (
    vector_store
    .similarity_search_by_vector(
        embedding=hyde_vector,
        k=4
    )
)


print(
    "\nHYDE RETRIEVAL RESULTS"
)

print(
    "=" * 80
)


for i, document in enumerate(
    hyde_documents,
    start=1
):

    print(
        f"\nResult {i}"
    )

    print(
        "Paper page:",
        document.metadata.get(
            "paper_page"
        )
    )

    print(
        "Section:",
        document.metadata.get(
            "section"
        )
    )

    print(
        "Chunk ID:",
        document.metadata.get(
            "chunk_id"
        )
    )

    print(
        "-" * 80
    )

    print(
        document.page_content[:1000]
    )



HYDE RETRIEVAL RESULTS

Result 1
Paper page: 3
Section: introduction
Chunk ID: llama2-page-3-chunk-10
--------------------------------------------------------------------------------
continue to improve the safety of those models, paving the way for more responsible development of LLMs.
We also share novel observations we made during the development ofLlama 2 and Llama 2-Chat, such as
the emergence of tool usage and temporal organization of knowledge.
3

Result 2
Paper page: 4
Section: introduction
Chunk ID: llama2-page-4-chunk-12
--------------------------------------------------------------------------------
1. Llama 2, an updated version ofLlama 1, trained on a new mix of publicly available data. We also
increased the size of the pretraining corpus by 40%, doubled the context length of the model, and
adopted grouped-query attention (Ainslie et al., 2023). We are releasing variants ofLlama 2 with
7B, 13B, and 70B parameters. We have also trained 34B variants, which we report on in t

### Learning: COMPARE NORMAL VS HYDE

**What you'll learn:** Embed a guessed answer, then search near that vector.

**What this cell does:** Runs `COMPARE NORMAL VS HYDE` and prints intermediate results you can inspect.

**Watch for:** Helps when the question vocabulary differs from the docs.



In [ ]:
# ============================================================
# 22. COMPARE NORMAL VS HYDE
# ============================================================

query = (
    "How does Llama 2 improve safety?"
)


### Learning: Normal Dense Retrieval

**What you'll learn:** Retrieve by semantic nearest-neighbors.

**What this cell does:** Executes retrieval/generation for: Normal Dense Retrieval.

**Watch for:** Dense can miss exact codes/IDs — compare with BM25 later.



In [ ]:
# ------------------------------------------------------------
# Normal Dense Retrieval
# ------------------------------------------------------------

normal_documents = (
    vector_store
    .similarity_search(
        query=query,
        k=4
    )
)

### Learning: HyDE Retrieval

**What you'll learn:** Create vectors that put similar meaning nearby.

**What this cell does:** Runs `HyDE Retrieval` and prints intermediate results you can inspect.

**Watch for:** Note dimension size; it must match the index.



In [ ]:
# ------------------------------------------------------------
# HyDE Retrieval
# ------------------------------------------------------------

hyde_vector = (
    hyde_embeddings
    .embed_query(
        query
    )
)


### Learning: hyde_documents = (

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Executes retrieval/generation for: hyde_documents = (.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
hyde_documents = (
    vector_store
    .similarity_search_by_vector(
        embedding=hyde_vector,
        k=4
    )
)

### Learning: PRINT COMPARISON

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `PRINT COMPARISON` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 23. PRINT COMPARISON
# ============================================================

print(
    "\nNORMAL DENSE RETRIEVAL"
)

print(
    "=" * 80
)

for i, doc in enumerate(
    normal_documents,
    start=1
):

    print(
        i,
        "| Page:",
        doc.metadata.get(
            "paper_page"
        ),
        "| Section:",
        doc.metadata.get(
            "section"
        ),
        "|",
        doc.page_content[:200]
        .replace(
            "\n",
            " "
        )
    )


print(
    "\nHYDE RETRIEVAL"
)

print(
    "=" * 80
)

for i, doc in enumerate(
    hyde_documents,
    start=1
):

    print(
        i,
        "| Page:",
        doc.metadata.get(
            "paper_page"
        ),
        "| Section:",
        doc.metadata.get(
            "section"
        ),
        "|",
        doc.page_content[:200]
        .replace(
            "\n",
            " "
        )
    )



NORMAL DENSE RETRIEVAL
1 | Page: 3 | Section: introduction | continue to improve the safety of those models, paving the way for more responsible development of LLMs. We also share novel observations we made during the development ofLlama 2 and Llama 2-Chat, suc
2 | Page: 3 | Section: introduction | the community to advance AI alignment research. In this work, we develop and release Llama 2, a family of pretrained and fine-tuned LLMs,Llama 2 and Llama 2-Chat, at scales up to 70B parameters. On th
3 | Page: 4 | Section: introduction | Figure 3: Safety human evaluation results forLlama 2-Chat compared to other open-source and closed- source models. Human raters judged model generations for safety violations across ~2,000 adversarial
4 | Page: 4 | Section: introduction | 1. Llama 2, an updated version ofLlama 1, trained on a new mix of publicly available data. We also increased the size of the pretraining corpus by 40%, doubled the context length of the model, and ado

HYDE RETRIEVAL
1 | 

### Learning: FINAL FLOW

**What you'll learn:** Retrieve by semantic nearest-neighbors.

**What this cell does:** Runs `FINAL FLOW` and prints intermediate results you can inspect.

**Watch for:** Dense can miss exact codes/IDs — compare with BM25 later.



In [ ]:
# ============================================================
# 24. FINAL FLOW
# ============================================================

"""
NORMAL DENSE RETRIEVAL

User Query
    ↓
Query Embedding
    ↓
Vector Search
    ↓
Relevant Documents


HYDE RETRIEVAL

User Query
    ↓
LLM generates hypothetical document
    ↓
Hypothetical document embedding
    ↓
Vector Search
    ↓
Relevant REAL documents
"""


print(
    "\nPractical completed successfully."
)

### Learning: hypothetical_document = hyde_embeddings.llm_chain.invoke(

**What you'll learn:** Create vectors that put similar meaning nearby.

**What this cell does:** Executes retrieval/generation for: hypothetical_document = hyde_embeddings.llm_chain.invoke(.

**Watch for:** Note dimension size; it must match the index.



In [ ]:
hypothetical_document = hyde_embeddings.llm_chain.invoke(
    {"query": query}
)

hyde_vector = hyde_embeddings.embed_query(query)

hyde_documents = vector_store.similarity_search_by_vector(
    embedding=hyde_vector,
    k=4
)

# multiquery- reteieval

### Learning: MULTI-QUERY RETRIEVER PRACTICAL

**What you'll learn:** Install the packages this notebook needs.

**What this cell does:** Runs `MULTI-QUERY RETRIEVER PRACTICAL` and prints intermediate results you can inspect.

**Watch for:** Run once; restart runtime if Colab asks.



In [ ]:
# ============================================================
# MULTI-QUERY RETRIEVER PRACTICAL
# ============================================================

# ------------------------------------------------------------
# 1. INSTALL REQUIRED PACKAGES
# ------------------------------------------------------------

# Run only if packages are not already installed
# %pip install -U langchain langchain-classic langchain-openai


# ============================================================
# 2. IMPORTS
# ============================================================

from langchain_openai import ChatOpenAI
from langchain_classic.retrievers.multi_query import (
    MultiQueryRetriever
)

### Learning: CREATE BASE RETRIEVER

**What you'll learn:** Build or reload the vector index used by retrievers.

**What this cell does:** Runs `CREATE BASE RETRIEVER` and prints intermediate results you can inspect.

**Watch for:** Use REBUILD_INDEX=False after the first successful build.



In [ ]:
# ============================================================
# 3. CREATE BASE RETRIEVER
# ============================================================

# We are using the existing Chroma vector_store
# created earlier in the notebook.

base_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    }
)

print("Base retriever created successfully.")

Base retriever created successfully.


### Learning: CREATE LLM FOR QUERY GENERATION

**What you'll learn:** Authenticate so embedding and chat calls can run.

**What this cell does:** Runs `CREATE LLM FOR QUERY GENERATION` and prints intermediate results you can inspect.

**Watch for:** Never hardcode secrets in shared notebooks.



In [ ]:
# ============================================================
# 4. CREATE LLM FOR QUERY GENERATION
# ============================================================

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

print("LLM created successfully.")

LLM created successfully.


### Learning: CREATE MULTI-QUERY RETRIEVER

**What you'll learn:** Ask the corpus several phrasings and pool the hits.

**What this cell does:** Runs `CREATE MULTI-QUERY RETRIEVER` and prints intermediate results you can inspect.

**Watch for:** Compare against a single dense query on the same question.



In [ ]:
# ============================================================
# 5. CREATE MULTI-QUERY RETRIEVER
# ============================================================

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm,
    include_original=True
)

print("MultiQueryRetriever created successfully.")


MultiQueryRetriever created successfully.


### Learning: USER QUERY

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Runs `USER QUERY` and prints intermediate results you can inspect.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
# ============================================================
# 6. USER QUERY
# ============================================================

query = (
    "How does Llama 2 improve safety?"
)

print("\nOriginal Query:")
print(query)



Original Query:
How does Llama 2 improve safety?


### Learning: RUN MULTI-QUERY RETRIEVER

**What you'll learn:** Ask the corpus several phrasings and pool the hits.

**What this cell does:** Executes retrieval/generation for: RUN MULTI-QUERY RETRIEVER.

**Watch for:** Compare against a single dense query on the same question.



In [ ]:
# ============================================================
# 8. RUN MULTI-QUERY RETRIEVER
# ============================================================

documents = multi_query_retriever.invoke(
    query
)

### Learning: documents

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Runs `documents` and prints intermediate results you can inspect.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
documents

[Document(id='7efcf997-1a34-4035-b867-888e9ea699a6', metadata={'paper_page': 3, 'chunk_id': 'llama2-page-3-chunk-10', 'producer': 'pdfTeX-1.40.25', 'title': '', 'section': 'introduction', 'year': 2023, 'creationdate': '2023-07-20T00:30:36+00:00', 'subject': '', 'creator': 'LaTeX with hyperref', 'author': '', 'moddate': '2023-07-20T00:30:36+00:00', 'source': 'D:\\complete_content_new\\Full-Stack-GenAI-Bootcamp-1.0\\Class-37-08-Aug-2026-prompting\\data\\llama2-research-paper.pdf', 'total_pages': 77, 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'page': 2, 'keywords': '', 'organization': 'Meta', 'paper': 'Llama 2', 'access_level': 'public', 'start_index': 3414, 'trapped': '/False', 'page_label': '3', 'document_type': 'research_paper'}, page_content='continue to improve the safety of those models, paving the way for more responsible development of LLMs.\nWe also share novel observations we made during the development ofLlama 2 

### Learning: DISPLAY FINAL RETRIEVED DOCUMENTS

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `DISPLAY FINAL RETRIEVED DOCUMENTS` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 9. DISPLAY FINAL RETRIEVED DOCUMENTS
# ============================================================

print(
    "\nMULTI-QUERY RETRIEVAL RESULTS"
)

print(
    "=" * 100
)

print(
    f"Total unique documents returned: "
    f"{len(documents)}"
)


for i, document in enumerate(
    documents,
    start=1
):

    print(
        "\n" + "=" * 100
    )

    print(
        f"RESULT {i}"
    )

    print(
        "Paper page:",
        document.metadata.get(
            "paper_page"
        )
    )

    print(
        "Section:",
        document.metadata.get(
            "section"
        )
    )

    print(
        "Chunk ID:",
        document.metadata.get(
            "chunk_id"
        )
    )

    print(
        "-" * 100
    )

    print(
        document.page_content[:1000]
    )



MULTI-QUERY RETRIEVAL RESULTS
Total unique documents returned: 5

RESULT 1
Paper page: 3
Section: introduction
Chunk ID: llama2-page-3-chunk-10
----------------------------------------------------------------------------------------------------
continue to improve the safety of those models, paving the way for more responsible development of LLMs.
We also share novel observations we made during the development ofLlama 2 and Llama 2-Chat, such as
the emergence of tool usage and temporal organization of knowledge.
3

RESULT 2
Paper page: 3
Section: introduction
Chunk ID: llama2-page-3-chunk-9
----------------------------------------------------------------------------------------------------
the community to advance AI alignment research.
In this work, we develop and release Llama 2, a family of pretrained and fine-tuned LLMs,Llama 2 and
Llama 2-Chat, at scales up to 70B parameters. On the series of helpfulness and safety benchmarks we tested,
Llama 2-Chat models generally perform bette

### Learning: NORMAL RETRIEVAL FOR COMPARISON

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Executes retrieval/generation for: NORMAL RETRIEVAL FOR COMPARISON.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 10. NORMAL RETRIEVAL FOR COMPARISON
# ============================================================

normal_documents = base_retriever.invoke(
    query
)


print(
    "\n\nNORMAL DENSE RETRIEVAL"
)

print(
    "=" * 100
)

for i, document in enumerate(
    normal_documents,
    start=1
):

    print(
        f"\nResult {i}"
    )

    print(
        "Page:",
        document.metadata.get(
            "paper_page"
        )
    )

    print(
        "Section:",
        document.metadata.get(
            "section"
        )
    )

    print(
        "Chunk ID:",
        document.metadata.get(
            "chunk_id"
        )
    )

    print(
        document.page_content[:500]
    )



NORMAL DENSE RETRIEVAL

Result 1
Page: 3
Section: introduction
Chunk ID: llama2-page-3-chunk-10
continue to improve the safety of those models, paving the way for more responsible development of LLMs.
We also share novel observations we made during the development ofLlama 2 and Llama 2-Chat, such as
the emergence of tool usage and temporal organization of knowledge.
3

Result 2
Page: 3
Section: introduction
Chunk ID: llama2-page-3-chunk-9
the community to advance AI alignment research.
In this work, we develop and release Llama 2, a family of pretrained and fine-tuned LLMs,Llama 2 and
Llama 2-Chat, at scales up to 70B parameters. On the series of helpfulness and safety benchmarks we tested,
Llama 2-Chat models generally perform better than existing open-source models. They also appear to
be on par with some of the closed-source models, at least on the human evaluations we performed (see
Figures 1 and 3). We have taken measures to

Result 3
Page: 4
Section: introduction
Chunk ID: llam

### Learning: MULTI-QUERY RESULTS FOR COMPARISON

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `MULTI-QUERY RESULTS FOR COMPARISON` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:

# ============================================================
# 11. MULTI-QUERY RESULTS FOR COMPARISON
# ============================================================

print(
    "\n\nMULTI-QUERY RETRIEVAL"
)

print(
    "=" * 100
)

for i, document in enumerate(
    documents,
    start=1
):

    print(
        f"\nResult {i}"
    )

    print(
        "Page:",
        document.metadata.get(
            "paper_page"
        )
    )

    print(
        "Section:",
        document.metadata.get(
            "section"
        )
    )

    print(
        "Chunk ID:",
        document.metadata.get(
            "chunk_id"
        )
    )

    print(
        document.page_content[:500]
    )




MULTI-QUERY RETRIEVAL

Result 1
Page: 3
Section: introduction
Chunk ID: llama2-page-3-chunk-10
continue to improve the safety of those models, paving the way for more responsible development of LLMs.
We also share novel observations we made during the development ofLlama 2 and Llama 2-Chat, such as
the emergence of tool usage and temporal organization of knowledge.
3

Result 2
Page: 3
Section: introduction
Chunk ID: llama2-page-3-chunk-9
the community to advance AI alignment research.
In this work, we develop and release Llama 2, a family of pretrained and fine-tuned LLMs,Llama 2 and
Llama 2-Chat, at scales up to 70B parameters. On the series of helpfulness and safety benchmarks we tested,
Llama 2-Chat models generally perform better than existing open-source models. They also appear to
be on par with some of the closed-source models, at least on the human evaluations we performed (see
Figures 1 and 3). We have taken measures to

Result 3
Page: 4
Section: introduction
Chunk ID: llama

### Learning: FINAL FLOW

**What you'll learn:** Ask the corpus several phrasings and pool the hits.

**What this cell does:** Runs `FINAL FLOW` and prints intermediate results you can inspect.

**Watch for:** Compare against a single dense query on the same question.



In [ ]:
# ============================================================
# 12. FINAL FLOW
# ============================================================

"""
NORMAL RETRIEVAL

User Query
    ↓
Embedding
    ↓
Vector Search
    ↓
Top-K Documents


MULTI-QUERY RETRIEVAL

User Query
    ↓
LLM generates multiple query variations
    ↓
Query 1 ──→ Retriever ──→ Documents
Query 2 ──→ Retriever ──→ Documents
Query 3 ──→ Retriever ──→ Documents
Original ─→ Retriever ──→ Documents
    ↓
Merge all results
    ↓
Remove duplicate documents
    ↓
Final unique documents
"""


print(
    "\nMulti-Query Retriever practical completed successfully."
)

### Learning: base_retriever = vector_store.as_retriever(

**What you'll learn:** Retrieve by semantic nearest-neighbors.

**What this cell does:** Executes retrieval/generation for: base_retriever = vector_store.as_retriever(.

**Watch for:** Dense can miss exact codes/IDs — compare with BM25 later.



In [ ]:
base_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm,
    include_original=True
)

documents = multi_query_retriever.invoke(
    "How does Llama 2 improve safety?"
)

Original Query
      ↓
LLM generates different versions
      ↓
Q1 → Vector Search
Q2 → Vector Search
Q3 → Vector Search
Original → Vector Search
      ↓
Combine all documents
      ↓
Remove duplicates
      ↓
Final Results

### Learning: sentence window retriever practical

**What you'll learn:** Match a sentence, return the surrounding window for context.

**What this cell does:** Runs `sentence window retriever practical` and prints intermediate results you can inspect.

**Watch for:** Precision of the match + context of the neighborhood.



In [ ]:
# sentence window retriever practical

### Learning: SENTENCE WINDOW RETRIEVAL PRACTICAL

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** Runs `SENTENCE WINDOW RETRIEVAL PRACTICAL` and prints intermediate results you can inspect.

**Watch for:** If an import fails, re-run the install cell.



In [ ]:
# ============================================================
# SENTENCE WINDOW RETRIEVAL PRACTICAL
# ============================================================

# ------------------------------------------------------------
# 1. IMPORTS
# ------------------------------------------------------------

import re
import uuid

from langchain_core.documents import Document
from langchain_chroma import Chroma


### Learning: CONFIGURATION

**What you'll learn:** Build or reload the vector index used by retrievers.

**What this cell does:** Runs `CONFIGURATION` and prints intermediate results you can inspect.

**Watch for:** Use REBUILD_INDEX=False after the first successful build.



In [ ]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

# Number of sentences before and after the matched sentence
WINDOW_SIZE = 2

SENTENCE_WINDOW_COLLECTION = "sentence_window_retriever_demo"

SENTENCE_WINDOW_PERSIST_DIR = (
    DATA_DIR / "sentence_window_chroma"
)


### Learning: SENTENCE SPLITTING FUNCTION

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Defines helper logic for: SENTENCE SPLITTING FUNCTION.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
# ============================================================
# 3. SENTENCE SPLITTING FUNCTION
# ============================================================

def split_into_sentences(text: str) -> list[str]:
    """
    Simple sentence splitter.

    Splits text after:
    .
    !
    ?

    while keeping reasonably clean sentence boundaries.
    """

    text = text.strip()

    if not text:
        return []

    sentences = re.split(
        r'(?<=[.!?])\s+',
        text
    )

    return [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]


### Learning: CREATE SENTENCE-WINDOW DOCUMENTS

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Runs `CREATE SENTENCE-WINDOW DOCUMENTS` and prints intermediate results you can inspect.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
# ============================================================
# 4. CREATE SENTENCE-WINDOW DOCUMENTS
# ============================================================

sentence_documents = []


for page_document in pages:

    sentences = split_into_sentences(
        page_document.page_content
    )

    for sentence_index, sentence in enumerate(sentences):

        # --------------------------------------------
        # Calculate surrounding sentence range
        # --------------------------------------------

        start_index = max(
            0,
            sentence_index - WINDOW_SIZE
        )

        end_index = min(
            len(sentences),
            sentence_index + WINDOW_SIZE + 1
        )

        # --------------------------------------------
        # Build surrounding context window
        # --------------------------------------------

        window_sentences = sentences[
            start_index:end_index
        ]

        sentence_window = " ".join(
            window_sentences
        )

        # --------------------------------------------
        # Copy original metadata
        # --------------------------------------------

        metadata = dict(
            page_document.metadata
        )

        metadata.update(
            {
                "sentence_index": sentence_index,
                "window_start": start_index,
                "window_end": end_index - 1,
                "sentence_window": sentence_window,
                "original_sentence": sentence,
                "window_size": WINDOW_SIZE,
            }
        )

        # --------------------------------------------
        # IMPORTANT:
        # page_content contains ONLY the sentence.
        #
        # This small sentence is what gets embedded
        # and searched.
        # --------------------------------------------

        sentence_document = Document(
            page_content=sentence,
            metadata=metadata
        )

        sentence_documents.append(
            sentence_document
        )


print(
    f"Total sentence documents created: "
    f"{len(sentence_documents)}"
)


Total sentence documents created: 2910


### Learning: INSPECT ONE SENTENCE DOCUMENT

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `INSPECT ONE SENTENCE DOCUMENT` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:


# ============================================================
# 5. INSPECT ONE SENTENCE DOCUMENT
# ============================================================

example_document = sentence_documents[20]

print("\nSEARCHABLE SENTENCE:")
print(
    example_document.page_content
)

print("\nSURROUNDING WINDOW:")
print(
    example_document.metadata[
        "sentence_window"
    ]
)

print("\nMETADATA:")
print(
    example_document.metadata
)



SEARCHABLE SENTENCE:
.

SURROUNDING WINDOW:
. . . . .

METADATA:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\complete_content_new\\Full-Stack-GenAI-Bootcamp-1.0\\Class-37-08-Aug-2026-prompting\\data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 1, 'page_label': '2', 'paper': 'Llama 2', 'organization': 'Meta', 'year': 2023, 'document_type': 'research_paper', 'paper_page': 2, 'section': 'front_matter', 'access_level': 'public', 'sentence_index': 14, 'window_start': 12, 'window_end': 16, 'sentence_window': '. . . . .', 'original_sentence': '.', 'window_size': 2}


### Learning: CREATE SENTENCE-LEVEL VECTOR STORE

**What you'll learn:** Create vectors that put similar meaning nearby.

**What this cell does:** Runs `CREATE SENTENCE-LEVEL VECTOR STORE` and prints intermediate results you can inspect.

**Watch for:** Note dimension size; it must match the index.



In [ ]:
# ============================================================
# 6. CREATE SENTENCE-LEVEL VECTOR STORE
# ============================================================

sentence_vector_store = Chroma.from_documents(
    documents=sentence_documents,
    embedding=embeddings,
    collection_name=SENTENCE_WINDOW_COLLECTION,
    persist_directory=str(
        SENTENCE_WINDOW_PERSIST_DIR
    ),
)


print(
    "\nSentence-level vector store created."
)


Sentence-level vector store created.


### Learning: CREATE BASE SENTENCE RETRIEVER

**What you'll learn:** Retrieve by semantic nearest-neighbors.

**What this cell does:** Runs `CREATE BASE SENTENCE RETRIEVER` and prints intermediate results you can inspect.

**Watch for:** Dense can miss exact codes/IDs — compare with BM25 later.



In [ ]:
# ============================================================
# 7. CREATE BASE SENTENCE RETRIEVER
# ============================================================

sentence_retriever = (
    sentence_vector_store
    .as_retriever(
        search_type="similarity",
        search_kwargs={
            "k": 4
        }
    )
)


print(
    "Sentence retriever created successfully."
)


Sentence retriever created successfully.


### Learning: USER QUERY

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Runs `USER QUERY` and prints intermediate results you can inspect.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
# ============================================================
# 8. USER QUERY
# ============================================================

query = (
    "How was Llama 2 aligned using human feedback?"
)

print("\nUSER QUERY:")
print(query)


USER QUERY:
How was Llama 2 aligned using human feedback?


### Learning: RETRIEVE MATCHING SENTENCES

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Executes retrieval/generation for: RETRIEVE MATCHING SENTENCES.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 9. RETRIEVE MATCHING SENTENCES
# ============================================================

matched_sentences = (
    sentence_retriever.invoke(
        query
    )
)


print(
    "\nMATCHED SENTENCES"
)

print(
    "=" * 100
)


for i, document in enumerate(
    matched_sentences,
    start=1
):

    print(
        f"\nMATCH {i}"
    )

    print(
        "Page:",
        document.metadata.get(
            "paper_page"
        )
    )

    print(
        "Sentence index:",
        document.metadata.get(
            "sentence_index"
        )
    )

    print(
        "\nMatched sentence:"
    )

    print(
        document.page_content
    )


MATCHED SENTENCES

MATCH 1
Page: 8
Sentence index: 19

Matched sentence:
3 Fine-tuning
Llama 2-Chat is the result of several months of research and iterative applications of alignment techniques,
including both instruction tuning and RLHF, requiring significant computational and annotation resources.

MATCH 2
Page: 10
Sentence index: 26

Matched sentence:
Leveraging such response scores as rewards, we can optimizeLlama 2-Chat during RLHF for
better human preference alignment and improved helpfulness and safety.

MATCH 3
Page: 4
Sentence index: 6

Matched sentence:
Llama 2, an updated version ofLlama 1, trained on a new mix of publicly available data.

MATCH 4
Page: 3
Sentence index: 0

Matched sentence:
Figure 1: Helpfulness human evaluationresults forLlama
2-Chatcompared to other open-source and closed-source
models.


### Learning: REPLACE MATCHED SENTENCE WITH SENTENCE WINDOW

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `REPLACE MATCHED SENTENCE WITH SENTENCE WINDOW` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 10. REPLACE MATCHED SENTENCE WITH SENTENCE WINDOW
# ============================================================

window_documents = []


for matched_document in matched_sentences:

    window_text = (
        matched_document
        .metadata
        .get(
            "sentence_window",
            matched_document.page_content
        )
    )

    window_document = Document(
        page_content=window_text,
        metadata={
            **matched_document.metadata,

            "matched_sentence":
                matched_document.page_content,

            "retrieval_type":
                "sentence_window"
        }
    )

    window_documents.append(
        window_document
    )


### Learning: DISPLAY SENTENCE WINDOW RESULTS

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `DISPLAY SENTENCE WINDOW RESULTS` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 11. DISPLAY SENTENCE WINDOW RESULTS
# ============================================================

print(
    "\n\nSENTENCE WINDOW RESULTS"
)

print(
    "=" * 100
)


for i, document in enumerate(
    window_documents,
    start=1
):

    print(
        f"\nRESULT {i}"
    )

    print(
        "Paper page:",
        document.metadata.get(
            "paper_page"
        )
    )

    print(
        "Sentence index:",
        document.metadata.get(
            "sentence_index"
        )
    )

    print(
        "\nMatched sentence:"
    )

    print(
        document.metadata.get(
            "matched_sentence"
        )
    )

    print(
        "\nReturned sentence window:"
    )

    print(
        document.page_content
    )

    print(
        "\n" + "-" * 100
    )



SENTENCE WINDOW RESULTS

RESULT 1
Paper page: 8
Sentence index: 19

Matched sentence:
3 Fine-tuning
Llama 2-Chat is the result of several months of research and iterative applications of alignment techniques,
including both instruction tuning and RLHF, requiring significant computational and annotation resources.

Returned sentence window:
Results for the
PaLM-2-L are from Anil et al. (2023). 3 Fine-tuning
Llama 2-Chat is the result of several months of research and iterative applications of alignment techniques,
including both instruction tuning and RLHF, requiring significant computational and annotation resources. In this section, we report on our experiments and findings using supervised fine-tuning (Section 3.1), as
well as initial and iterative reward modeling (Section 3.2.2) and RLHF (Section 3.2.3). We also share a
new technique, Ghost Attention (GAtt), which we find helps control dialogue flow over multiple turns
(Section 3.3).

----------------------------------------------

### Learning: CREATE A REUSABLE SENTENCE WINDOW RETRIEVER FUNCTION

**What you'll learn:** Create vectors that put similar meaning nearby.

**What this cell does:** Defines helper logic for: CREATE A REUSABLE SENTENCE WINDOW RETRIEVER FUNCTION.

**Watch for:** Note dimension size; it must match the index.



In [ ]:
# ============================================================
# 12. CREATE A REUSABLE SENTENCE WINDOW RETRIEVER FUNCTION
# ============================================================

def sentence_window_retrieve(
    query: str,
    k: int = 4
):
    """
    Search using individual sentence embeddings,
    but return each matched sentence together with
    its surrounding context window.
    """

    # --------------------------------------------
    # Retrieve the best matching sentences
    # --------------------------------------------

    retriever = (
        sentence_vector_store
        .as_retriever(
            search_type="similarity",
            search_kwargs={
                "k": k
            }
        )
    )

    matched_documents = (
        retriever.invoke(
            query
        )
    )

    # --------------------------------------------
    # Expand each sentence into its context window
    # --------------------------------------------

    expanded_documents = []

    for document in matched_documents:

        window_text = (
            document.metadata.get(
                "sentence_window",
                document.page_content
            )
        )

        expanded_document = Document(
            page_content=window_text,
            metadata={
                **document.metadata,

                "matched_sentence":
                    document.page_content,

                "retrieval_type":
                    "sentence_window"
            }
        )

        expanded_documents.append(
            expanded_document
        )

    return expanded_documents


### Learning: TEST THE REUSABLE FUNCTION

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `TEST THE REUSABLE FUNCTION` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 13. TEST THE REUSABLE FUNCTION
# ============================================================

query = (
    "How does reinforcement learning improve Llama 2-Chat?"
)

results = sentence_window_retrieve(
    query=query,
    k=4
)


print(
    "\nREUSABLE SENTENCE WINDOW RETRIEVER"
)

print(
    "=" * 100
)


for i, document in enumerate(
    results,
    start=1
):

    print(
        f"\nRESULT {i}"
    )

    print(
        "Page:",
        document.metadata.get(
            "paper_page"
        )
    )

    print(
        "\nMatched sentence:"
    )

    print(
        document.metadata.get(
            "matched_sentence"
        )
    )

    print(
        "\nFull sentence window:"
    )

    print(
        document.page_content
    )

    print(
        "\n" + "-" * 100
    )



REUSABLE SENTENCE WINDOW RETRIEVER

RESULT 1
Page: 13

Matched sentence:
Therefore, everything else being equal, an improvement of the reward model can be directly translated into
an improvement forLlama 2-Chat.

Full sentence window:
We note that reward model accuracy is one of the most
important proxies for the final performance ofLlama 2-Chat. While best practices for comprehensively
evaluating a generative model is an open research question, the ranking task of the reward has no ambiguity. Therefore, everything else being equal, an improvement of the reward model can be directly translated into
an improvement forLlama 2-Chat. 3.2.3 Iterative Fine-Tuning
As we received more batches of human preference data annotation, we were able to train better reward
models and collect more prompts. We therefore trained successive versions for RLHF models, referred to
here as RLHF-V1, ..., RLHF-V5.

-------------------------------------------------------------------------------------------------

### Learning: COMPARE NORMAL SENTENCE SEARCH VS SENTENCE WINDOW

**What you'll learn:** Match a sentence, return the surrounding window for context.

**What this cell does:** Executes retrieval/generation for: COMPARE NORMAL SENTENCE SEARCH VS SENTENCE WINDOW.

**Watch for:** Precision of the match + context of the neighborhood.



In [ ]:
# ============================================================
# 14. COMPARE NORMAL SENTENCE SEARCH VS SENTENCE WINDOW
# ============================================================

query = (
    "How was human preference data "
    "used to improve Llama 2?"
)


# ------------------------------------------------------------
# Normal sentence retrieval
# ------------------------------------------------------------

normal_sentence_results = (
    sentence_retriever.invoke(
        query
    )
)


# ------------------------------------------------------------
# Sentence window retrieval
# ------------------------------------------------------------

sentence_window_results = (
    sentence_window_retrieve(
        query=query,
        k=4
    )
)


### Learning: DISPLAY COMPARISON

**What you'll learn:** Match a sentence, return the surrounding window for context.

**What this cell does:** Runs `DISPLAY COMPARISON` and prints intermediate results you can inspect.

**Watch for:** Precision of the match + context of the neighborhood.



In [ ]:
# ============================================================
# 15. DISPLAY COMPARISON
# ============================================================

print(
    "\nNORMAL SENTENCE RETRIEVAL"
)

print(
    "=" * 100
)


for i, document in enumerate(
    normal_sentence_results,
    start=1
):

    print(
        f"\nResult {i}:"
    )

    print(
        document.page_content
    )


print(
    "\n\nSENTENCE WINDOW RETRIEVAL"
)

print(
    "=" * 100
)


for i, document in enumerate(
    sentence_window_results,
    start=1
):

    print(
        f"\nResult {i}:"
    )

    print(
        document.page_content
    )



NORMAL SENTENCE RETRIEVAL

Result 1:
As we collected more preference data, our
reward models improved, and we were able to train progressively better versions forLlama 2-Chat (see
the results in Section 5, Figure 20).Llama 2-Chat improvement also shifted the model’s data distribution.

Result 2:
Llama 2, an updated version ofLlama 1, trained on a new mix of publicly available data.

Result 3:
Leveraging such response scores as rewards, we can optimizeLlama 2-Chat during RLHF for
better human preference alignment and improved helpfulness and safety.

Result 4:
This reflects the nature of our
iterative model update and preference data annotation procedure - with better-performingLlama 2-Chat
models used for response sampling over time, it becomes challenging for annotators to select a better one
from two equally high-quality responses.


SENTENCE WINDOW RETRIEVAL

Result 1:
Safety guidelines and more detailed information regarding safety annotations
can be found in Section 4.2.1. Human 

### Learning: FINAL CONCEPTUAL FLOW

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `FINAL CONCEPTUAL FLOW` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 16. FINAL CONCEPTUAL FLOW
# ============================================================

"""
SENTENCE WINDOW RETRIEVAL

Document
    ↓
Split into individual sentences
    ↓
For every sentence:
    store surrounding sentences in metadata
    ↓
Embed ONLY individual sentences
    ↓
User Query
    ↓
Vector Search
    ↓
Best matching sentence
    ↓
Read sentence_window from metadata
    ↓
Return:
previous sentences
+
matched sentence
+
next sentences
    ↓
LLM
"""


print(
    "\nSentence Window Retrieval "
    "practical completed successfully."
)

### Learning: Sentence 8 → Llama 2 first undergoes supervised fine-tuning.

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Runs `Sentence 8 → Llama 2 first undergoes supervised fine-tuning.` and prints intermediate results you can inspect.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
Sentence 8 → Llama 2 first undergoes supervised fine-tuning.
Sentence 9 → Human preference data is collected.
Sentence 10 → RLHF is used to align Llama 2-Chat.
Sentence 11 → Reward models score candidate responses.
Sentence 12 → PPO is used for optimization.

### Learning: Sentence 10

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Runs `Sentence 10` and prints intermediate results you can inspect.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
Sentence 10
→ "RLHF is used to align Llama 2-Chat."

### Learning: {

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Runs `{` and prints intermediate results you can inspect.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
{
    "sentence_window":
    """
    Llama 2 first undergoes supervised fine-tuning.
    Human preference data is collected.
    RLHF is used to align Llama 2-Chat.
    Reward models score candidate responses.
    PPO is used for optimization.
    """
}

### Learning: How is Llama 2 aligned using human feedback?

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Runs `How is Llama 2 aligned using human feedback?` and prints intermediate results you can inspect.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
How is Llama 2 aligned using human feedback?

### Learning: Sentence 10

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Runs `Sentence 10` and prints intermediate results you can inspect.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
Sentence 10

### Learning: Sentence 8

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Runs `Sentence 8` and prints intermediate results you can inspect.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
Sentence 8
Sentence 9
Sentence 10
Sentence 11
Sentence 12

### Learning: parent document retriever practical

**What you'll learn:** Search small child chunks, return larger parent context.

**What this cell does:** Runs `parent document retriever practical` and prints intermediate results you can inspect.

**Watch for:** Child finds; parent explains — don't confuse the two stores.



In [ ]:
# parent document retriever practical

### Learning: PARENT DOCUMENT RETRIEVER PRACTICAL

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** Runs `PARENT DOCUMENT RETRIEVER PRACTICAL` and prints intermediate results you can inspect.

**Watch for:** If an import fails, re-run the install cell.



In [ ]:
# ============================================================
# PARENT DOCUMENT RETRIEVER PRACTICAL
# ============================================================

# ------------------------------------------------------------
# 1. IMPORTS
# ------------------------------------------------------------

from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.stores import InMemoryStore

### Learning: CONFIGURATION

**What you'll learn:** Build or reload the vector index used by retrievers.

**What this cell does:** Runs `CONFIGURATION` and prints intermediate results you can inspect.

**Watch for:** Use REBUILD_INDEX=False after the first successful build.



In [ ]:

# ============================================================
# 2. CONFIGURATION
# ============================================================

PARENT_COLLECTION_NAME = "parent_document_retriever_demo"

PARENT_VECTORSTORE_DIR = (
    DATA_DIR / "parent_document_chroma"
)


### Learning: CREATE PARENT SPLITTER

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Runs `CREATE PARENT SPLITTER` and prints intermediate results you can inspect.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
# ============================================================
# 3. CREATE PARENT SPLITTER
# ============================================================

# Large chunks that will finally be returned to the user/LLM

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200
)



### Learning: CREATE CHILD SPLITTER

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Runs `CREATE CHILD SPLITTER` and prints intermediate results you can inspect.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
# ============================================================
# 4. CREATE CHILD SPLITTER
# ============================================================

# Small chunks used for embedding and similarity search

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50
)


print("Parent and child splitters created.")


Parent and child splitters created.


### Learning: CREATE EMPTY VECTOR STORE

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Runs `CREATE EMPTY VECTOR STORE` and prints intermediate results you can inspect.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
# ============================================================
# 5. CREATE EMPTY VECTOR STORE
# ============================================================

# IMPORTANT:
# Child chunks will be stored here.

parent_vector_store = Chroma(
    collection_name=PARENT_COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(
        PARENT_VECTORSTORE_DIR
    )
)


print("Child vector store created.")


Child vector store created.


### Learning: CREATE DOCUMENT STORE

**What you'll learn:** Search small child chunks, return larger parent context.

**What this cell does:** Runs `CREATE DOCUMENT STORE` and prints intermediate results you can inspect.

**Watch for:** Child finds; parent explains — don't confuse the two stores.



In [ ]:
# ============================================================
# 6. CREATE DOCUMENT STORE
# ============================================================

# IMPORTANT:
# Parent documents are stored separately here.

docstore = InMemoryStore()


print("Parent document store created.")

Parent document store created.


### Learning: CREATE PARENT DOCUMENT RETRIEVER

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Runs `CREATE PARENT DOCUMENT RETRIEVER` and prints intermediate results you can inspect.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
# ============================================================
# 7. CREATE PARENT DOCUMENT RETRIEVER
# ============================================================

parent_retriever = ParentDocumentRetriever(
    vectorstore=parent_vector_store,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)


print(
    "ParentDocumentRetriever created successfully."
)



ParentDocumentRetriever created successfully.


### Learning: ADD ORIGINAL DOCUMENTS

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** Runs `ADD ORIGINAL DOCUMENTS` and prints intermediate results you can inspect.

**Watch for:** Confirm page count and first-page text look sane.



In [ ]:
# ============================================================
# 8. ADD ORIGINAL DOCUMENTS
# ============================================================

# "pages" comes from your previous PyPDFLoader code.
#
# Internally:
#
# Original Pages
#      ↓
# Parent Splitter
#      ↓
# Large Parent Chunks
#      ↓
# Child Splitter
#      ↓
# Small Child Chunks
#
# Child Chunks  → Vector Store
# Parent Chunks → Docstore

parent_retriever.add_documents(
    pages
)


print(
    "Documents added to ParentDocumentRetriever."
)

Documents added to ParentDocumentRetriever.


### Learning: CHECK CHILD CHUNK COUNT

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Runs `CHECK CHILD CHUNK COUNT` and prints intermediate results you can inspect.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
# ============================================================
# 9. CHECK CHILD CHUNK COUNT
# ============================================================

child_count = (
    parent_vector_store
    ._collection
    .count()
)

print(
    f"Total child chunks stored in vector DB: "
    f"{child_count}"
)


Total child chunks stored in vector DB: 856


### Learning: USER QUERY

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Runs `USER QUERY` and prints intermediate results you can inspect.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
# ============================================================
# 10. USER QUERY
# ============================================================

query = (
    "How was Llama 2 trained using human feedback?"
)


print("\nUSER QUERY:")
print(query)




USER QUERY:
How was Llama 2 trained using human feedback?


### Learning: RETRIEVE PARENT DOCUMENTS

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Executes retrieval/generation for: RETRIEVE PARENT DOCUMENTS.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 11. RETRIEVE PARENT DOCUMENTS
# ============================================================

parent_documents = (
    parent_retriever.invoke(
        query
    )
)


print(
    "\nPARENT DOCUMENT RETRIEVAL RESULTS"
)

print(
    "=" * 100
)


for i, document in enumerate(
    parent_documents,
    start=1
):

    print(
        f"\nRESULT {i}"
    )

    print(
        "Paper page:",
        document.metadata.get(
            "paper_page"
        )
    )

    print(
        "Section:",
        document.metadata.get(
            "section"
        )
    )

    print(
        "Returned document length:",
        len(document.page_content)
    )

    print(
        "-" * 100
    )

    print(
        document.page_content[:1500]
    )



PARENT DOCUMENT RETRIEVAL RESULTS

RESULT 1
Paper page: 5
Section: pretraining
Returned document length: 1923
----------------------------------------------------------------------------------------------------
Figure 4: Training ofLlama 2-Chat: This process begins with thepretraining of Llama 2 using publicly
available online sources. Following this, we create an initial version ofLlama 2-Chatthrough the application
of supervised fine-tuning. Subsequently, the model is iteratively refined using Reinforcement Learning
with Human Feedback(RLHF) methodologies, specifically through rejection sampling and Proximal Policy
Optimization (PPO). Throughout the RLHF stage, the accumulation ofiterative reward modeling datain
parallel with model enhancements is crucial to ensure the reward models remain within distribution.
2 Pretraining
Tocreatethenewfamilyof Llama 2models,webeganwiththepretrainingapproachdescribedinTouvronetal.
(2023), using an optimized auto-regressive transformer, but made se

### Learning: DIRECT CHILD-CHUNK SEARCH

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Executes retrieval/generation for: DIRECT CHILD-CHUNK SEARCH.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 12. DIRECT CHILD-CHUNK SEARCH
# ============================================================

# This directly searches the underlying vector store.
# These are the SMALL chunks that actually match the query.

child_documents = (
    parent_vector_store
    .similarity_search(
        query=query,
        k=4
    )
)


print(
    "\n\nDIRECT CHILD-CHUNK SEARCH"
)

print(
    "=" * 100
)


for i, document in enumerate(
    child_documents,
    start=1
):

    print(
        f"\nCHILD RESULT {i}"
    )

    print(
        "Child length:",
        len(document.page_content)
    )

    print(
        "Parent ID:",
        document.metadata.get(
            "doc_id"
        )
    )

    print(
        "-" * 100
    )

    print(
        document.page_content
    )



DIRECT CHILD-CHUNK SEARCH

CHILD RESULT 1
Child length: 312
Parent ID: 0541ba9d-32db-498f-acd8-c9279e66c499
----------------------------------------------------------------------------------------------------
Figure 4: Training ofLlama 2-Chat: This process begins with thepretraining of Llama 2 using publicly
available online sources. Following this, we create an initial version ofLlama 2-Chatthrough the application
of supervised fine-tuning. Subsequently, the model is iteratively refined using Reinforcement Learning

CHILD RESULT 2
Child length: 313
Parent ID: e81d3afc-48b2-4ab0-96b4-4cd51be89832
----------------------------------------------------------------------------------------------------
reward models improved, and we were able to train progressively better versions forLlama 2-Chat (see
the results in Section 5, Figure 20).Llama 2-Chat improvement also shifted the model’s data distribution.
Since reward model accuracy can quickly degrade if not exposed to this new sample dist

### Learning: COMPARE CHILD VS PARENT

**What you'll learn:** Search small child chunks, return larger parent context.

**What this cell does:** Runs `COMPARE CHILD VS PARENT` and prints intermediate results you can inspect.

**Watch for:** Child finds; parent explains — don't confuse the two stores.



In [ ]:
# ============================================================
# 13. COMPARE CHILD VS PARENT
# ============================================================

print(
    "\n\nCHILD VS PARENT COMPARISON"
)

print(
    "=" * 100
)


print("\nCHILD RESULTS:")

for i, document in enumerate(
    child_documents,
    start=1
):

    print(
        f"{i}. Length = "
        f"{len(document.page_content)}"
    )


print("\nPARENT RESULTS:")

for i, document in enumerate(
    parent_documents,
    start=1
):

    print(
        f"{i}. Length = "
        f"{len(document.page_content)}"
    )




CHILD VS PARENT COMPARISON

CHILD RESULTS:
1. Length = 312
2. Length = 313
3. Length = 334
4. Length = 316

PARENT RESULTS:
1. Length = 1923
2. Length = 1999
3. Length = 1255


### Learning: INSPECT CHILD → PARENT RELATIONSHIP

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `INSPECT CHILD → PARENT RELATIONSHIP` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 14. INSPECT CHILD → PARENT RELATIONSHIP
# ============================================================

print(
    "\n\nCHILD TO PARENT IDs"
)

print(
    "=" * 100
)


for i, child in enumerate(
    child_documents,
    start=1
):

    parent_id = child.metadata.get(
        "doc_id"
    )

    print(
        f"\nChild {i}"
    )

    print(
        "Parent ID:",
        parent_id
    )

    print(
        "Child content:"
    )

    print(
        child.page_content[:300]
    )





CHILD TO PARENT IDs

Child 1
Parent ID: 0541ba9d-32db-498f-acd8-c9279e66c499
Child content:
Figure 4: Training ofLlama 2-Chat: This process begins with thepretraining of Llama 2 using publicly
available online sources. Following this, we create an initial version ofLlama 2-Chatthrough the application
of supervised fine-tuning. Subsequently, the model is iteratively refined using Reinforcem

Child 2
Parent ID: e81d3afc-48b2-4ab0-96b4-4cd51be89832
Child content:
reward models improved, and we were able to train progressively better versions forLlama 2-Chat (see
the results in Section 5, Figure 20).Llama 2-Chat improvement also shifted the model’s data distribution.
Since reward model accuracy can quickly degrade if not exposed to this new sample distributio

Child 3
Parent ID: 87bf0868-82ca-403b-a00c-c4d7dfdecce5
Child content:
3 Fine-tuning
Llama 2-Chat is the result of several months of research and iterative applications of alignment techniques,
including both instruction tuning and

### Learning: FETCH A PARENT DIRECTLY FROM DOCSTORE

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `FETCH A PARENT DIRECTLY FROM DOCSTORE` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 15. FETCH A PARENT DIRECTLY FROM DOCSTORE
# ============================================================

# Take the parent ID of the first matching child.

if child_documents:

    parent_id = (
        child_documents[0]
        .metadata
        .get("doc_id")
    )

    if parent_id:

        stored_parent = (
            docstore.mget(
                [parent_id]
            )[0]
        )

        print(
            "\n\nPARENT FETCHED DIRECTLY FROM DOCSTORE"
        )

        print(
            "=" * 100
        )

        print(
            "Parent ID:",
            parent_id
        )

        print(
            "\nParent length:",
            len(
                stored_parent.page_content
            )
        )

        print(
            "\nParent content:"
        )

        print(
            stored_parent.page_content[:2000]
        )



PARENT FETCHED DIRECTLY FROM DOCSTORE
Parent ID: 0541ba9d-32db-498f-acd8-c9279e66c499

Parent length: 1923

Parent content:
Figure 4: Training ofLlama 2-Chat: This process begins with thepretraining of Llama 2 using publicly
available online sources. Following this, we create an initial version ofLlama 2-Chatthrough the application
of supervised fine-tuning. Subsequently, the model is iteratively refined using Reinforcement Learning
with Human Feedback(RLHF) methodologies, specifically through rejection sampling and Proximal Policy
Optimization (PPO). Throughout the RLHF stage, the accumulation ofiterative reward modeling datain
parallel with model enhancements is crucial to ensure the reward models remain within distribution.
2 Pretraining
Tocreatethenewfamilyof Llama 2models,webeganwiththepretrainingapproachdescribedinTouvronetal.
(2023), using an optimized auto-regressive transformer, but made several changes to improve performance.
Specifically, we performed more robust data clea

### Learning: CREATE REUSABLE FUNCTION

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Defines helper logic for: CREATE REUSABLE FUNCTION.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
# ============================================================
# 16. CREATE REUSABLE FUNCTION
# ============================================================

def parent_document_search(
    query: str
):
    """
    Search small child chunks,
    but return their larger parent chunks.
    """

    documents = (
        parent_retriever.invoke(
            query
        )
    )

    return documents

### Learning: TEST REUSABLE FUNCTION

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `TEST REUSABLE FUNCTION` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 17. TEST REUSABLE FUNCTION
# ============================================================

query = (
    "What safety techniques were "
    "used for Llama 2-Chat?"
)

results = parent_document_search(
    query
)


print(
    "\n\nREUSABLE PARENT DOCUMENT RETRIEVER"
)

print(
    "=" * 100
)


for i, document in enumerate(
    results,
    start=1
):

    print(
        f"\nRESULT {i}"
    )

    print(
        "Page:",
        document.metadata.get(
            "paper_page"
        )
    )

    print(
        "Section:",
        document.metadata.get(
            "section"
        )
    )

    print(
        "Length:",
        len(document.page_content)
    )

    print(
        document.page_content[:1200]
    )





REUSABLE PARENT DOCUMENT RETRIEVER

RESULT 1
Page: 24
Section: safety
Length: 1932
advice). The attack vectors explored consist of psychological manipulation (e.g., authority manipulation),
logic manipulation (e.g., false premises), syntactic manipulation (e.g., misspelling), semantic manipulation
(e.g., metaphor), perspective manipulation (e.g., role playing), non-English languages, and others.
Wethendefinebestpracticesforsafeandhelpfulmodelresponses: themodelshouldfirstaddressimmediate
safetyconcernsifapplicable,thenaddressthepromptbyexplainingthepotentialriskstotheuser,andfinally
provide additional information if possible. We also ask the annotators to avoid negative user experience
categories (see Appendix A.5.2). The guidelines are meant to be a general guide for the model and are
iteratively refined and revised to include newly identified risks.
4.2.2 Safety Supervised Fine-Tuning
In accordance with the established guidelines from Section 4.2.1, we gather prompts and demonstrat

### Learning: FINAL CONCEPTUAL FLOW

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `FINAL CONCEPTUAL FLOW` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 18. FINAL CONCEPTUAL FLOW
# ============================================================

"""
PARENT DOCUMENT RETRIEVAL

Original Document
        ↓
Parent Splitter
        ↓
Large Parent Chunks
        ↓
Child Splitter
        ↓
Small Child Chunks
        ↓
Create Embeddings for CHILD chunks
        ↓
Store CHILD chunks in Vector DB
        ↓
Store PARENT chunks in Docstore
        ↓

User Query
        ↓
Query Embedding
        ↓
Search CHILD chunks
        ↓
Best child chunk found
        ↓
Read parent ID from child metadata
        ↓
Fetch corresponding PARENT from Docstore
        ↓
Return larger parent context
        ↓
LLM
"""


print(
    "\nParent Document Retriever "
    "practical completed successfully."
)


Parent Document Retriever practical completed successfully.


### Learning: parent_retriever = ParentDocumentRetriever(

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Executes retrieval/generation for: parent_retriever = ParentDocumentRetriever(.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
parent_retriever = ParentDocumentRetriever(
    vectorstore=parent_vector_store,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

parent_retriever.add_documents(pages)

documents = parent_retriever.invoke(
    "How was Llama 2 trained using human feedback?"
)

### Learning: Original Document

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Runs `Original Document` and prints intermediate results you can inspect.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
# Original Document
#         ↓
# Large Parent Chunk
#         ↓
# Small Child Chunks
#         ↓
# Child Embeddings
#         ↓
# Vector Search
#         ↓
# Matched Child
#         ↓
# Parent ID
#         ↓
# Parent Document
#         ↓
# Final Return

### Learning: Child chunks  → Vector DB

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Runs `Child chunks  → Vector DB` and prints intermediate results you can inspect.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
Child chunks  → Vector DB
Parent chunks → Docstore

Search Child
Return Parent

##### multihop retriever practical

### Learning: MULTI-HOP RETRIEVAL PRACTICAL

**What you'll learn:** Install the packages this notebook needs.

**What this cell does:** Runs `MULTI-HOP RETRIEVAL PRACTICAL` and prints intermediate results you can inspect.

**Watch for:** Run once; restart runtime if Colab asks.



In [ ]:
# ============================================================
# MULTI-HOP RETRIEVAL PRACTICAL
# LangChain + LangGraph
# ============================================================

# ------------------------------------------------------------
# 1. INSTALL PACKAGES
# ------------------------------------------------------------

# %pip install -U langchain langchain-openai langgraph


# ============================================================
# 2. IMPORTS
# ============================================================

from typing import TypedDict, List

from langchain_openai import ChatOpenAI
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import (
    StateGraph,
    START,
    END,
)


### Learning: CREATE BASE RETRIEVER

**What you'll learn:** Build or reload the vector index used by retrievers.

**What this cell does:** Runs `CREATE BASE RETRIEVER` and prints intermediate results you can inspect.

**Watch for:** Use REBUILD_INDEX=False after the first successful build.



In [ ]:


# ============================================================
# 3. CREATE BASE RETRIEVER
# ============================================================

# Existing Chroma vector_store from previous practical

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    }
)

print("Base retriever created.")


### Learning: CREATE LLM

**What you'll learn:** Authenticate so embedding and chat calls can run.

**What this cell does:** Runs `CREATE LLM` and prints intermediate results you can inspect.

**Watch for:** Never hardcode secrets in shared notebooks.



In [ ]:
# ============================================================
# 4. CREATE LLM
# ============================================================

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

print("LLM created.")


### Learning: DEFINE GRAPH STATE

**What you'll learn:** Use evidence from hop 1 to invent a better hop-2 query.

**What this cell does:** Runs `DEFINE GRAPH STATE` and prints intermediate results you can inspect.

**Watch for:** Multi-hop is intentional; default RAG does one trip only.



In [ ]:

# ============================================================
# 5. DEFINE GRAPH STATE
# ============================================================

class MultiHopState(TypedDict):

    original_query: str

    hop1_query: str

    hop1_documents: List[Document]

    hop1_summary: str

    hop2_query: str

    hop2_documents: List[Document]

    final_answer: str


### Learning: HOP 1 RETRIEVAL

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Defines helper logic for: HOP 1 RETRIEVAL.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 6. HOP 1 RETRIEVAL
# ============================================================

def hop1_retrieve(
    state: MultiHopState
):

    query = state["original_query"]

    print("\n" + "=" * 100)
    print("HOP 1 QUERY:")
    print(query)

    documents = retriever.invoke(
        query
    )

    print("\nHOP 1 DOCUMENTS:")

    for i, doc in enumerate(
        documents,
        start=1
    ):

        print(
            f"\nDocument {i}"
        )

        print(
            "Page:",
            doc.metadata.get(
                "paper_page"
            )
        )

        print(
            doc.page_content[:500]
        )

    return {
        "hop1_query": query,
        "hop1_documents": documents
    }


### Learning: SUMMARIZE HOP 1 EVIDENCE

**What you'll learn:** Use evidence from hop 1 to invent a better hop-2 query.

**What this cell does:** Runs `SUMMARIZE HOP 1 EVIDENCE` and prints intermediate results you can inspect.

**Watch for:** Multi-hop is intentional; default RAG does one trip only.



In [ ]:
# ============================================================
# 7. SUMMARIZE HOP 1 EVIDENCE
# ============================================================

hop1_summary_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are helping with multi-hop retrieval.

Read the retrieved context and extract only the
information useful for determining what should
be searched in the next retrieval hop.

Do not answer the final user question yet.
"""
        ),
        (
            "human",
            """
Original question:

{query}


Retrieved context:

{context}
"""
        )
    ]
)


### Learning: def summarize_hop1(

**What you'll learn:** Use evidence from hop 1 to invent a better hop-2 query.

**What this cell does:** Defines helper logic for: def summarize_hop1(.

**Watch for:** Multi-hop is intentional; default RAG does one trip only.



In [ ]:

def summarize_hop1(
    state: MultiHopState
):

    context = "\n\n".join(
        doc.page_content
        for doc in state[
            "hop1_documents"
        ]
    )

    response = llm.invoke(
        hop1_summary_prompt.format_messages(
            query=state[
                "original_query"
            ],
            context=context
        )
    )

    summary = response.content

    print("\n" + "=" * 100)
    print("HOP 1 SUMMARY:")
    print(summary)

    return {
        "hop1_summary": summary
    }



### Learning: GENERATE SECOND-HOP QUERY

**What you'll learn:** Use evidence from hop 1 to invent a better hop-2 query.

**What this cell does:** Runs `GENERATE SECOND-HOP QUERY` and prints intermediate results you can inspect.

**Watch for:** Multi-hop is intentional; default RAG does one trip only.



In [ ]:
# ============================================================
# 8. GENERATE SECOND-HOP QUERY
# ============================================================

next_query_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are generating the next search query
for a multi-hop retrieval system.

Based on:

1. the original user question
2. the evidence retrieved in Hop 1

generate ONE new search query that retrieves
the missing information needed to answer the
original question.

Rules:

- Do not answer the final question.
- Return only the new search query.
- The query must be standalone.
- Use information discovered in Hop 1.
"""
        ),
        (
            "human",
            """
Original question:

{original_query}


Hop 1 evidence:

{hop1_summary}
"""
        )
    ]
)


### Learning: def generate_hop2_query(

**What you'll learn:** Use evidence from hop 1 to invent a better hop-2 query.

**What this cell does:** Defines helper logic for: def generate_hop2_query(.

**Watch for:** Multi-hop is intentional; default RAG does one trip only.



In [ ]:
def generate_hop2_query(
    state: MultiHopState
):

    response = llm.invoke(
        next_query_prompt.format_messages(
            original_query=state[
                "original_query"
            ],
            hop1_summary=state[
                "hop1_summary"
            ]
        )
    )

    hop2_query = (
        response.content.strip()
    )

    print("\n" + "=" * 100)
    print("GENERATED HOP 2 QUERY:")
    print(hop2_query)

    return {
        "hop2_query": hop2_query
    }

### Learning: HOP 2 RETRIEVAL

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Defines helper logic for: HOP 2 RETRIEVAL.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 9. HOP 2 RETRIEVAL
# ============================================================

def hop2_retrieve(
    state: MultiHopState
):

    query = state[
        "hop2_query"
    ]

    documents = retriever.invoke(
        query
    )

    print("\n" + "=" * 100)
    print("HOP 2 DOCUMENTS:")

    for i, doc in enumerate(
        documents,
        start=1
    ):

        print(
            f"\nDocument {i}"
        )

        print(
            "Page:",
            doc.metadata.get(
                "paper_page"
            )
        )

        print(
            doc.page_content[:500]
        )

    return {
        "hop2_documents": documents
    }


### Learning: GENERATE FINAL ANSWER

**What you'll learn:** Use evidence from hop 1 to invent a better hop-2 query.

**What this cell does:** Runs `GENERATE FINAL ANSWER` and prints intermediate results you can inspect.

**Watch for:** Multi-hop is intentional; default RAG does one trip only.



In [ ]:
# ============================================================
# 10. GENERATE FINAL ANSWER
# ============================================================

final_answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
Answer the user's question using only the
evidence retrieved during Hop 1 and Hop 2.

If the evidence is insufficient, say that the
retrieved documents are insufficient.

Do not invent information.
"""
        ),
        (
            "human",
            """
Original question:

{query}


Hop 1 evidence:

{hop1_context}


Hop 2 evidence:

{hop2_context}
"""
        )
    ]
)

### Learning: def generate_final_answer(

**What you'll learn:** Use evidence from hop 1 to invent a better hop-2 query.

**What this cell does:** Defines helper logic for: def generate_final_answer(.

**Watch for:** Multi-hop is intentional; default RAG does one trip only.



In [ ]:
def generate_final_answer(
    state: MultiHopState
):

    hop1_context = "\n\n".join(
        doc.page_content
        for doc in state[
            "hop1_documents"
        ]
    )

    hop2_context = "\n\n".join(
        doc.page_content
        for doc in state[
            "hop2_documents"
        ]
    )

    response = llm.invoke(
        final_answer_prompt.format_messages(
            query=state[
                "original_query"
            ],
            hop1_context=hop1_context,
            hop2_context=hop2_context
        )
    )

    final_answer = (
        response.content
    )

    print("\n" + "=" * 100)
    print("FINAL ANSWER:")
    print(final_answer)

    return {
        "final_answer":
            final_answer
    }


### Learning: BUILD LANGGRAPH

**What you'll learn:** Use evidence from hop 1 to invent a better hop-2 query.

**What this cell does:** Runs `BUILD LANGGRAPH` and prints intermediate results you can inspect.

**Watch for:** Multi-hop is intentional; default RAG does one trip only.



In [ ]:
# ============================================================
# 11. BUILD LANGGRAPH
# ============================================================

builder = StateGraph(
    MultiHopState
)


# Add nodes

builder.add_node(
    "hop1_retrieve",
    hop1_retrieve
)

builder.add_node(
    "summarize_hop1",
    summarize_hop1
)

builder.add_node(
    "generate_hop2_query",
    generate_hop2_query
)

builder.add_node(
    "hop2_retrieve",
    hop2_retrieve
)

builder.add_node(
    "generate_final_answer",
    generate_final_answer
)


### Learning: CONNECT GRAPH

**What you'll learn:** Use evidence from hop 1 to invent a better hop-2 query.

**What this cell does:** Runs `CONNECT GRAPH` and prints intermediate results you can inspect.

**Watch for:** Multi-hop is intentional; default RAG does one trip only.



In [ ]:
# ============================================================
# 12. CONNECT GRAPH
# ============================================================

builder.add_edge(
    START,
    "hop1_retrieve"
)

builder.add_edge(
    "hop1_retrieve",
    "summarize_hop1"
)

builder.add_edge(
    "summarize_hop1",
    "generate_hop2_query"
)

builder.add_edge(
    "generate_hop2_query",
    "hop2_retrieve"
)

builder.add_edge(
    "hop2_retrieve",
    "generate_final_answer"
)

builder.add_edge(
    "generate_final_answer",
    END
)


### Learning: COMPILE GRAPH

**What you'll learn:** Use evidence from hop 1 to invent a better hop-2 query.

**What this cell does:** Executes retrieval/generation for: COMPILE GRAPH.

**Watch for:** Multi-hop is intentional; default RAG does one trip only.



In [ ]:
# ============================================================
# 13. COMPILE GRAPH
# ============================================================

multi_hop_graph = (
    builder.compile()
)

print(
    "\nMulti-Hop Retrieval Graph created successfully."
)


# ============================================================
# 14. TEST QUERY
# ============================================================

query = (
    "How was Llama 2-Chat aligned with human preferences "
    "and what role did reward models play in that process?"
)


result = multi_hop_graph.invoke(
    {
        "original_query": query
    }
)


### Learning: FINAL OUTPUT

**What you'll learn:** Use evidence from hop 1 to invent a better hop-2 query.

**What this cell does:** Runs `FINAL OUTPUT` and prints intermediate results you can inspect.

**Watch for:** Multi-hop is intentional; default RAG does one trip only.



In [ ]:
# ============================================================
# 15. FINAL OUTPUT
# ============================================================

print("\n" + "=" * 100)
print("ORIGINAL QUERY:")
print(
    result["original_query"]
)

print("\n" + "=" * 100)
print("HOP 1 QUERY:")
print(
    result["hop1_query"]
)

print("\n" + "=" * 100)
print("HOP 1 SUMMARY:")
print(
    result["hop1_summary"]
)

print("\n" + "=" * 100)
print("HOP 2 QUERY:")
print(
    result["hop2_query"]
)

print("\n" + "=" * 100)
print("FINAL ANSWER:")
print(
    result["final_answer"]
)

### Learning: FINAL CONCEPTUAL FLOW

**What you'll learn:** Use evidence from hop 1 to invent a better hop-2 query.

**What this cell does:** Runs `FINAL CONCEPTUAL FLOW` and prints intermediate results you can inspect.

**Watch for:** Multi-hop is intentional; default RAG does one trip only.



In [ ]:
# ============================================================
# 16. FINAL CONCEPTUAL FLOW
# ============================================================

"""
MULTI-HOP RETRIEVAL

Original User Query
        ↓
Hop 1 Retrieval
        ↓
Retrieve first evidence
        ↓
Analyze / summarize evidence
        ↓
Generate next query
        ↓
Hop 2 Retrieval
        ↓
Retrieve additional evidence
        ↓
Combine Hop 1 + Hop 2 evidence
        ↓
LLM
        ↓
Final Answer
"""


print(
    "\nMulti-Hop Retrieval practical completed successfully."
)

Suppose query hai:

How was Llama 2-Chat aligned with human preferences
and what role did reward models play?

Hop 1 may search:

How was Llama 2-Chat aligned with human preferences?

and retrieve RLHF-related chunks.

Then LLM sees those chunks and realizes:

I also need information about reward models.

So it generates Hop 2 query:

How were reward models trained and used in Llama 2-Chat?

Then:

Hop 2 Query
     ↓
Retriever
     ↓
Reward-model documents

Finally:

Hop 1 evidence
       +
Hop 2 evidence
       ↓
Final Answer

## Weighted fusion

### Learning: WEIGHTED RRF / HYBRID RETRIEVAL PRACTICAL

**What you'll learn:** Install the packages this notebook needs.

**What this cell does:** Runs `WEIGHTED RRF / HYBRID RETRIEVAL PRACTICAL` and prints intermediate results you can inspect.

**Watch for:** Run once; restart runtime if Colab asks.



In [ ]:
# ============================================================
# WEIGHTED RRF / HYBRID RETRIEVAL PRACTICAL
# LangChain EnsembleRetriever
# ============================================================

# ------------------------------------------------------------
# 1. INSTALL
# ------------------------------------------------------------

# %pip install -U langchain-classic langchain-community rank-bm25


# ============================================================
# 2. IMPORTS
# ============================================================

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

### Learning: CREATE SPARSE / BM25 RETRIEVER

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Runs `CREATE SPARSE / BM25 RETRIEVER` and prints intermediate results you can inspect.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
# ============================================================
# 3. CREATE SPARSE / BM25 RETRIEVER
# ============================================================

bm25_retriever = BM25Retriever.from_documents(
    chunks
)

bm25_retriever.k = 5

print("BM25 retriever created.")

### Learning: CREATE DENSE / VECTOR RETRIEVER

**What you'll learn:** Retrieve by semantic nearest-neighbors.

**What this cell does:** Runs `CREATE DENSE / VECTOR RETRIEVER` and prints intermediate results you can inspect.

**Watch for:** Dense can miss exact codes/IDs — compare with BM25 later.



In [ ]:
# ============================================================
# 4. CREATE DENSE / VECTOR RETRIEVER
# ============================================================

dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5
    }
)

print("Dense retriever created.")


### Learning: CREATE WEIGHTED HYBRID RETRIEVER

**What you'll learn:** Match exact tokens with lexical sparse retrieval.

**What this cell does:** Runs `CREATE WEIGHTED HYBRID RETRIEVER` and prints intermediate results you can inspect.

**Watch for:** Best for IDs, acronyms, and rare proper nouns.



In [ ]:
# ============================================================
# 5. CREATE WEIGHTED HYBRID RETRIEVER
# ============================================================

# IMPORTANT:
#
# LangChain EnsembleRetriever performs
# Weighted Reciprocal Rank Fusion (Weighted RRF).
#
# Here:
# BM25 weight  = 0.4
# Dense weight = 0.6

weighted_hybrid_retriever = EnsembleRetriever(
    retrievers=[
        bm25_retriever,
        dense_retriever
    ],
    weights=[
        0.4,
        0.6
    ]
)

print(
    "Weighted hybrid retriever created."
)


### Learning: USER QUERY

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Runs `USER QUERY` and prints intermediate results you can inspect.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
# ============================================================
# 6. USER QUERY
# ============================================================

query = (
    "How does Llama 2 improve safety?"
)

### Learning: BM25 RESULTS

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Executes retrieval/generation for: BM25 RESULTS.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 7. BM25 RESULTS
# ============================================================

bm25_results = bm25_retriever.invoke(
    query
)

print(
    "\nBM25 RESULTS"
)

print(
    "=" * 100
)

for i, document in enumerate(
    bm25_results,
    start=1
):

    print(
        f"\nRank {i}"
    )

    print(
        "Page:",
        document.metadata.get(
            "paper_page"
        )
    )

    print(
        "Chunk ID:",
        document.metadata.get(
            "chunk_id"
        )
    )

    print(
        document.page_content[:500]
    )
print("\nUSER QUERY:")
print(query)

### Learning: DENSE RESULTS

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Executes retrieval/generation for: DENSE RESULTS.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 8. DENSE RESULTS
# ============================================================

dense_results = dense_retriever.invoke(
    query
)

print(
    "\n\nDENSE VECTOR RESULTS"
)

print(
    "=" * 100
)

for i, document in enumerate(
    dense_results,
    start=1
):

    print(
        f"\nRank {i}"
    )

    print(
        "Page:",
        document.metadata.get(
            "paper_page"
        )
    )

    print(
        "Chunk ID:",
        document.metadata.get(
            "chunk_id"
        )
    )

    print(
        document.page_content[:500]
    )

### Learning: WEIGHTED RRF RESULTS

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Executes retrieval/generation for: WEIGHTED RRF RESULTS.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 9. WEIGHTED RRF RESULTS
# ============================================================

hybrid_results = (
    weighted_hybrid_retriever.invoke(
        query
    )
)

print(
    "\n\nWEIGHTED RRF RESULTS"
)

print(
    "=" * 100
)

for i, document in enumerate(
    hybrid_results,
    start=1
):

    print(
        f"\nFINAL RANK {i}"
    )

    print(
        "Page:",
        document.metadata.get(
            "paper_page"
        )
    )

    print(
        "Section:",
        document.metadata.get(
            "section"
        )
    )

    print(
        "Chunk ID:",
        document.metadata.get(
            "chunk_id"
        )
    )

    print(
        "-" * 100
    )

    print(
        document.page_content[:700]
    )


### Learning: FINAL FLOW

**What you'll learn:** Match exact tokens with lexical sparse retrieval.

**What this cell does:** Runs `FINAL FLOW` and prints intermediate results you can inspect.

**Watch for:** Best for IDs, acronyms, and rare proper nouns.



In [ ]:
# ============================================================
# 10. FINAL FLOW
# ============================================================

"""
                     USER QUERY
                         |
             -------------------------
             |                       |
             v                       v
       BM25 Retriever          Dense Retriever
             |                       |
             v                       v
        Ranked List A           Ranked List B
             |                       |
             -----------+-----------
                        |
                        v
               Weighted RRF

            BM25 Weight  = 0.4
            Dense Weight = 0.6

                        |
                        v
                Final Ranking
"""


print(
    "\nWeighted RRF practical completed."
)

### Learning: 2. Try Different Weights

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Runs `2. Try Different Weights` and prints intermediate results you can inspect.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
2. Try Different Weights

This part is useful in class:

### Learning: TEST DIFFERENT WEIGHT COMBINATIONS

**What you'll learn:** Match exact tokens with lexical sparse retrieval.

**What this cell does:** Executes retrieval/generation for: TEST DIFFERENT WEIGHT COMBINATIONS.

**Watch for:** Best for IDs, acronyms, and rare proper nouns.



In [ ]:
# ============================================================
# TEST DIFFERENT WEIGHT COMBINATIONS
# ============================================================

weight_configs = {
    "50_50": [0.5, 0.5],
    "70_BM25_30_Dense": [0.7, 0.3],
    "30_BM25_70_Dense": [0.3, 0.7],
}


for config_name, weights in weight_configs.items():

    retriever = EnsembleRetriever(
        retrievers=[
            bm25_retriever,
            dense_retriever
        ],
        weights=weights
    )

    results = retriever.invoke(
        query
    )

    print(
        "\n" + "=" * 100
    )

    print(
        f"CONFIGURATION: {config_name}"
    )

    print(
        f"BM25 weight: {weights[0]}"
    )

    print(
        f"Dense weight: {weights[1]}"
    )

    print(
        "=" * 100
    )

    for rank, document in enumerate(
        results[:5],
        start=1
    ):

        print(
            rank,
            "| Page:",
            document.metadata.get(
                "paper_page"
            ),
            "| Chunk:",
            document.metadata.get(
                "chunk_id"
            )
        )

### Learning: TRUE SCORE-BASED WEIGHTED FUSION

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** Runs `TRUE SCORE-BASED WEIGHTED FUSION` and prints intermediate results you can inspect.

**Watch for:** If an import fails, re-run the install cell.



In [ ]:
# ============================================================
# TRUE SCORE-BASED WEIGHTED FUSION
# ============================================================

import numpy as np


# ============================================================
# 1. QUERY
# ============================================================

query = (
    "How does Llama 2 improve safety?"
)


### Learning: DENSE RESULTS WITH SCORES

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Executes retrieval/generation for: DENSE RESULTS WITH SCORES.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
# ============================================================
# 2. DENSE RESULTS WITH SCORES
# ============================================================

dense_scored_results = (
    vector_store
    .similarity_search_with_relevance_scores(
        query=query,
        k=10
    )
)


### Learning: BUILD BM25 RETRIEVER INTERNALLY

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** Runs `BUILD BM25 RETRIEVER INTERNALLY` and prints intermediate results you can inspect.

**Watch for:** If an import fails, re-run the install cell.



In [ ]:
# ============================================================
# 3. BUILD BM25 RETRIEVER INTERNALLY
# ============================================================

from rank_bm25 import BM25Okapi


texts = [
    chunk.page_content
    for chunk in chunks
]


tokenized_corpus = [
    text.lower().split()
    for text in texts
]


bm25_model = BM25Okapi(
    tokenized_corpus
)


tokenized_query = (
    query.lower().split()
)


bm25_scores = bm25_model.get_scores(
    tokenized_query
)


### Learning: NORMALIZATION FUNCTION

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Defines helper logic for: NORMALIZATION FUNCTION.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
# ============================================================
# 4. NORMALIZATION FUNCTION
# ============================================================

def min_max_normalize(
    values
):

    values = np.asarray(
        values,
        dtype=float
    )

    minimum = values.min()
    maximum = values.max()

    if maximum == minimum:
        return np.zeros_like(
            values
        )

    return (
        values - minimum
    ) / (
        maximum - minimum
    )


### Learning: NORMALIZE BM25 SCORES

**What you'll learn:** Match exact tokens with lexical sparse retrieval.

**What this cell does:** Runs `NORMALIZE BM25 SCORES` and prints intermediate results you can inspect.

**Watch for:** Best for IDs, acronyms, and rare proper nouns.



In [ ]:
# ============================================================
# 5. NORMALIZE BM25 SCORES
# ============================================================

normalized_bm25_scores = (
    min_max_normalize(
        bm25_scores
    )
)


### Learning: CREATE SCORE MAP

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `CREATE SCORE MAP` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 6. CREATE SCORE MAP
# ============================================================

bm25_score_map = {}


for chunk, score in zip(
    chunks,
    normalized_bm25_scores
):

    chunk_id = chunk.metadata.get(
        "chunk_id"
    )

    bm25_score_map[
        chunk_id
    ] = float(
        score
    )


### Learning: COLLECT DENSE CANDIDATES

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `COLLECT DENSE CANDIDATES` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 7. COLLECT DENSE CANDIDATES
# ============================================================

candidate_rows = []


for document, dense_score in (
    dense_scored_results
):

    chunk_id = (
        document.metadata.get(
            "chunk_id"
        )
    )

    bm25_score = (
        bm25_score_map.get(
            chunk_id,
            0.0
        )
    )

    candidate_rows.append(
        {
            "document": document,
            "chunk_id": chunk_id,
            "bm25_score": bm25_score,
            "dense_score": float(
                dense_score
            )
        }
    )

### Learning: APPLY WEIGHTS

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Runs `APPLY WEIGHTS` and prints intermediate results you can inspect.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
# ============================================================
# 8. APPLY WEIGHTS
# ============================================================

BM25_WEIGHT = 0.4
DENSE_WEIGHT = 0.6


for row in candidate_rows:

    row["final_score"] = (
        BM25_WEIGHT
        * row["bm25_score"]
        +
        DENSE_WEIGHT
        * row["dense_score"]
    )


### Learning: SORT BY FINAL SCORE

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Runs `SORT BY FINAL SCORE` and prints intermediate results you can inspect.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
# ============================================================
# 9. SORT BY FINAL SCORE
# ============================================================

candidate_rows = sorted(
    candidate_rows,
    key=lambda x: x[
        "final_score"
    ],
    reverse=True
)


### Learning: DISPLAY RESULTS

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Runs `DISPLAY RESULTS` and prints intermediate results you can inspect.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:


# ============================================================
# 10. DISPLAY RESULTS
# ============================================================

print(
    "\nTRUE WEIGHTED SCORE FUSION"
)

print(
    "=" * 100
)


for rank, row in enumerate(
    candidate_rows,
    start=1
):

    document = row[
        "document"
    ]

    print(
        f"\nRank {rank}"
    )

    print(
        "Chunk ID:",
        row["chunk_id"]
    )

    print(
        f"BM25 normalized score: "
        f"{row['bm25_score']:.4f}"
    )

    print(
        f"Dense score: "
        f"{row['dense_score']:.4f}"
    )

    print(
        f"Final weighted score: "
        f"{row['final_score']:.4f}"
    )

    print(
        document.page_content[:500]
    )

## resiprocal rank fusion

### Learning: RECIPROCAL RANK FUSION (RRF) PRACTICAL

**What you'll learn:** Install the packages this notebook needs.

**What this cell does:** Runs `RECIPROCAL RANK FUSION (RRF) PRACTICAL` and prints intermediate results you can inspect.

**Watch for:** Run once; restart runtime if Colab asks.



In [ ]:
# ============================================================
# RECIPROCAL RANK FUSION (RRF) PRACTICAL
# LangChain EnsembleRetriever
# ============================================================


# ------------------------------------------------------------
# 1. INSTALL REQUIRED PACKAGES
# ------------------------------------------------------------

# %pip install -U langchain-classic langchain-community rank-bm25


# ============================================================
# 2. IMPORTS
# ============================================================

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

### Learning: CREATE BM25 / SPARSE RETRIEVER

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Runs `CREATE BM25 / SPARSE RETRIEVER` and prints intermediate results you can inspect.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
# ============================================================
# 3. CREATE BM25 / SPARSE RETRIEVER
# ============================================================

bm25_retriever = BM25Retriever.from_documents(
    chunks
)

bm25_retriever.k = 5

print("BM25 Retriever created.")

### Learning: CREATE DENSE / VECTOR RETRIEVER

**What you'll learn:** Retrieve by semantic nearest-neighbors.

**What this cell does:** Runs `CREATE DENSE / VECTOR RETRIEVER` and prints intermediate results you can inspect.

**Watch for:** Dense can miss exact codes/IDs — compare with BM25 later.



In [ ]:
# ============================================================
# 4. CREATE DENSE / VECTOR RETRIEVER
# ============================================================

dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5
    }
)

print("Dense Retriever created.")

### Learning: CREATE RRF RETRIEVER

**What you'll learn:** Retrieve by semantic nearest-neighbors.

**What this cell does:** Runs `CREATE RRF RETRIEVER` and prints intermediate results you can inspect.

**Watch for:** Dense can miss exact codes/IDs — compare with BM25 later.



In [ ]:
# ============================================================
# 5. CREATE RRF RETRIEVER
# ============================================================

# LangChain EnsembleRetriever uses
# Weighted Reciprocal Rank Fusion internally.
#
# We are assigning equal weights so that
# BM25 and Dense retrieval contribute equally.

rrf_retriever = EnsembleRetriever(
    retrievers=[
        bm25_retriever,
        dense_retriever
    ],
    weights=[
        1.0,
        1.0
    ]
)

print("RRF Retriever created successfully.")


### Learning: USER QUERY

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Runs `USER QUERY` and prints intermediate results you can inspect.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
# ============================================================
# 6. USER QUERY
# ============================================================

query = (
    "How does Llama 2 improve safety?"
)

print("\nUSER QUERY:")
print(query)

### Learning: GET BM25 RESULTS

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Executes retrieval/generation for: GET BM25 RESULTS.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 7. GET BM25 RESULTS
# ============================================================

bm25_results = bm25_retriever.invoke(
    query
)


print(
    "\nBM25 RESULTS"
)

print(
    "=" * 100
)


for rank, document in enumerate(
    bm25_results,
    start=1
):

    print(
        f"\nRank {rank}"
    )

    print(
        "Page:",
        document.metadata.get(
            "paper_page"
        )
    )

    print(
        "Chunk ID:",
        document.metadata.get(
            "chunk_id"
        )
    )

    print(
        document.page_content[:500]
    )

### Learning: GET DENSE RESULTS

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Executes retrieval/generation for: GET DENSE RESULTS.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:



# ============================================================
# 8. GET DENSE RESULTS
# ============================================================

dense_results = dense_retriever.invoke(
    query
)


print(
    "\n\nDENSE RESULTS"
)

print(
    "=" * 100
)


for rank, document in enumerate(
    dense_results,
    start=1
):

    print(
        f"\nRank {rank}"
    )

    print(
        "Page:",
        document.metadata.get(
            "paper_page"
        )
    )

    print(
        "Chunk ID:",
        document.metadata.get(
            "chunk_id"
        )
    )

    print(
        document.page_content[:500]
    )

### Learning: RUN RRF

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Executes retrieval/generation for: RUN RRF.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:



# ============================================================
# 9. RUN RRF
# ============================================================

rrf_results = rrf_retriever.invoke(
    query
)


print(
    "\n\nRRF FINAL RESULTS"
)

print(
    "=" * 100
)


for rank, document in enumerate(
    rrf_results,
    start=1
):

    print(
        f"\nFINAL RANK {rank}"
    )

    print(
        "Page:",
        document.metadata.get(
            "paper_page"
        )
    )

    print(
        "Section:",
        document.metadata.get(
            "section"
        )
    )

    print(
        "Chunk ID:",
        document.metadata.get(
            "chunk_id"
        )
    )

    print(
        "-" * 100
    )

    print(
        document.page_content[:700]
    )


### Learning: FINAL FLOW

**What you'll learn:** Match exact tokens with lexical sparse retrieval.

**What this cell does:** Runs `FINAL FLOW` and prints intermediate results you can inspect.

**Watch for:** Best for IDs, acronyms, and rare proper nouns.



In [ ]:
# ============================================================
# 10. FINAL FLOW
# ============================================================

"""
                         USER QUERY
                             |
                  -----------------------
                  |                     |
                  v                     v
            BM25 Retriever        Dense Retriever
                  |                     |
                  v                     v
            Ranked List A          Ranked List B
                  |                     |
                  -----------+----------
                             |
                             v
                   Reciprocal Rank Fusion
                             |
                             v
                      Final Ranking
"""


print(
    "\nRRF practical completed successfully."
)

Retrieve full chunks
       ↓
LLM checks each chunk against query
       ↓
Extract only query-relevant content
       ↓
Return compressed documents

### Learning: CONTEXTUAL COMPRESSION RETRIEVER PRACTICAL

**What you'll learn:** Install the packages this notebook needs.

**What this cell does:** Runs `CONTEXTUAL COMPRESSION RETRIEVER PRACTICAL` and prints intermediate results you can inspect.

**Watch for:** Run once; restart runtime if Colab asks.



In [ ]:
# ============================================================
# CONTEXTUAL COMPRESSION RETRIEVER PRACTICAL
# LangChain + LLMChainExtractor
# ============================================================


# ------------------------------------------------------------
# 1. INSTALL REQUIRED PACKAGES
# ------------------------------------------------------------

# %pip install -U langchain-classic langchain-openai


# ============================================================
# 2. IMPORTS
# ============================================================

from langchain_openai import ChatOpenAI

from langchain_classic.retrievers import (
    ContextualCompressionRetriever
)

from langchain_classic.retrievers.document_compressors import (
    LLMChainExtractor
)


### Learning: CREATE BASE RETRIEVER

**What you'll learn:** Build or reload the vector index used by retrievers.

**What this cell does:** Runs `CREATE BASE RETRIEVER` and prints intermediate results you can inspect.

**Watch for:** Use REBUILD_INDEX=False after the first successful build.



In [ ]:
# ============================================================
# 3. CREATE BASE RETRIEVER
# ============================================================

# We use the existing Chroma vector_store.
#
# First, normal vector retrieval will fetch documents.
# After that, contextual compression will be applied.

base_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    }
)

print("Base retriever created successfully.")

Base retriever created successfully.


### Learning: CREATE LLM

**What you'll learn:** Authenticate so embedding and chat calls can run.

**What this cell does:** Runs `CREATE LLM` and prints intermediate results you can inspect.

**Watch for:** Never hardcode secrets in shared notebooks.



In [ ]:
# ============================================================
# 4. CREATE LLM
# ============================================================

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

print("LLM created successfully.")

LLM created successfully.


### Learning: CREATE LLM-BASED DOCUMENT COMPRESSOR

**What you'll learn:** Trim retrieved chunks to the spans that answer the query.

**What this cell does:** Runs `CREATE LLM-BASED DOCUMENT COMPRESSOR` and prints intermediate results you can inspect.

**Watch for:** Compression is lossy — check that dates/caveats survive.



In [ ]:

# ============================================================
# 5. CREATE LLM-BASED DOCUMENT COMPRESSOR
# ============================================================

# LLMChainExtractor extracts only the portions of
# retrieved documents that are relevant to the query.

compressor = LLMChainExtractor.from_llm(
    llm
)

print("LLMChainExtractor created successfully.")


LLMChainExtractor created successfully.


### Learning: CREATE CONTEXTUAL COMPRESSION RETRIEVER

**What you'll learn:** Trim retrieved chunks to the spans that answer the query.

**What this cell does:** Runs `CREATE CONTEXTUAL COMPRESSION RETRIEVER` and prints intermediate results you can inspect.

**Watch for:** Compression is lossy — check that dates/caveats survive.



In [ ]:
# ============================================================
# 6. CREATE CONTEXTUAL COMPRESSION RETRIEVER
# ============================================================

compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

print(
    "ContextualCompressionRetriever "
    "created successfully."
)


ContextualCompressionRetriever created successfully.


### Learning: USER QUERY

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Executes retrieval/generation for: USER QUERY.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 7. USER QUERY
# ============================================================

query = (
    "How does Llama 2 improve safety?"
)

print("\nUSER QUERY:")
print(query)


# ============================================================
# 8. NORMAL RETRIEVAL
# ============================================================

normal_documents = base_retriever.invoke(
    query
)


print(
    "\nNORMAL RETRIEVAL RESULTS"
)

print(
    "=" * 100
)


for i, document in enumerate(
    normal_documents,
    start=1
):

    print(
        f"\nDOCUMENT {i}"
    )

    print(
        "Page:",
        document.metadata.get(
            "paper_page"
        )
    )

    print(
        "Section:",
        document.metadata.get(
            "section"
        )
    )

    print(
        "Original length:",
        len(document.page_content)
    )

    print(
        "-" * 100
    )

    print(
        document.page_content
    )



USER QUERY:
How does Llama 2 improve safety?

NORMAL RETRIEVAL RESULTS

DOCUMENT 1
Page: 3
Section: introduction
Original length: 274
----------------------------------------------------------------------------------------------------
continue to improve the safety of those models, paving the way for more responsible development of LLMs.
We also share novel observations we made during the development ofLlama 2 and Llama 2-Chat, such as
the emergence of tool usage and temporal organization of knowledge.
3

DOCUMENT 2
Page: 3
Section: introduction
Original length: 978
----------------------------------------------------------------------------------------------------
the community to advance AI alignment research.
In this work, we develop and release Llama 2, a family of pretrained and fine-tuned LLMs,Llama 2 and
Llama 2-Chat, at scales up to 70B parameters. On the series of helpfulness and safety benchmarks we tested,
Llama 2-Chat models generally perform better than existing open-sour

### Learning: RUN CONTEXTUAL COMPRESSION

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Executes retrieval/generation for: RUN CONTEXTUAL COMPRESSION.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 9. RUN CONTEXTUAL COMPRESSION
# ============================================================

compressed_documents = (
    compression_retriever.invoke(
        query
    )
)


print(
    "\n\nCONTEXTUAL COMPRESSION RESULTS"
)

print(
    "=" * 100
)


for i, document in enumerate(
    compressed_documents,
    start=1
):

    print(
        f"\nCOMPRESSED DOCUMENT {i}"
    )

    print(
        "Page:",
        document.metadata.get(
            "paper_page"
        )
    )

    print(
        "Section:",
        document.metadata.get(
            "section"
        )
    )

    print(
        "Compressed length:",
        len(document.page_content)
    )

    print(
        "-" * 100
    )

    print(
        document.page_content
    )




CONTEXTUAL COMPRESSION RESULTS

COMPRESSED DOCUMENT 1
Page: 3
Section: introduction
Compressed length: 104
----------------------------------------------------------------------------------------------------
continue to improve the safety of those models, paving the way for more responsible development of LLMs.

COMPRESSED DOCUMENT 2
Page: 3
Section: introduction
Compressed length: 530
----------------------------------------------------------------------------------------------------
Llama 2-Chat models generally perform better than existing open-source models. They also appear to be on par with some of the closed-source models, at least on the human evaluations we performed (see Figures 1 and 3). We have taken measures to increase the safety of these models, using safety-specific data annotation and tuning, as well as conducting red-teaming and employing iterative evaluations. Additionally, this paper contributes a thorough description of our fine-tuning methodology and approach to

### Learning: COMPARE ORIGINAL VS COMPRESSED

**What you'll learn:** Trim retrieved chunks to the spans that answer the query.

**What this cell does:** Runs `COMPARE ORIGINAL VS COMPRESSED` and prints intermediate results you can inspect.

**Watch for:** Compression is lossy — check that dates/caveats survive.



In [ ]:
# ============================================================
# 10. COMPARE ORIGINAL VS COMPRESSED
# ============================================================

print(
    "\n\nORIGINAL VS COMPRESSED"
)

print(
    "=" * 100
)


print(
    "\nOriginal documents returned:",
    len(normal_documents)
)

print(
    "Compressed documents returned:",
    len(compressed_documents)
)


original_characters = sum(
    len(document.page_content)
    for document in normal_documents
)


compressed_characters = sum(
    len(document.page_content)
    for document in compressed_documents
)


print(
    "\nTotal original characters:",
    original_characters
)

print(
    "Total compressed characters:",
    compressed_characters
)




ORIGINAL VS COMPRESSED

Original documents returned: 4
Compressed documents returned: 4

Total original characters: 3093
Total compressed characters: 1521


### Learning: CALCULATE COMPRESSION REDUCTION

**What you'll learn:** Trim retrieved chunks to the spans that answer the query.

**What this cell does:** Runs `CALCULATE COMPRESSION REDUCTION` and prints intermediate results you can inspect.

**Watch for:** Compression is lossy — check that dates/caveats survive.



In [ ]:
# ============================================================
# 11. CALCULATE COMPRESSION REDUCTION
# ============================================================

if original_characters > 0:

    reduction_percentage = (
        (
            original_characters
            -
            compressed_characters
        )
        /
        original_characters
    ) * 100

    print(
        f"\nContext reduction: "
        f"{reduction_percentage:.2f}%"
    )



Context reduction: 50.82%


### Learning: SIDE-BY-SIDE COMPARISON

**What you'll learn:** Trim retrieved chunks to the spans that answer the query.

**What this cell does:** Runs `SIDE-BY-SIDE COMPARISON` and prints intermediate results you can inspect.

**Watch for:** Compression is lossy — check that dates/caveats survive.



In [ ]:
# ============================================================
# 12. SIDE-BY-SIDE COMPARISON
# ============================================================

print(
    "\n\nSIDE-BY-SIDE CONCEPTUAL COMPARISON"
)

print(
    "=" * 100
)


for i, document in enumerate(
    normal_documents,
    start=1
):

    print(
        f"\nORIGINAL DOCUMENT {i}"
    )

    print(
        "-" * 100
    )

    print(
        document.page_content[:1000]
    )


print(
    "\n\nAFTER CONTEXTUAL COMPRESSION"
)

print(
    "=" * 100
)


for i, document in enumerate(
    compressed_documents,
    start=1
):

    print(
        f"\nCOMPRESSED DOCUMENT {i}"
    )

    print(
        "-" * 100
    )

    print(
        document.page_content
    )




SIDE-BY-SIDE CONCEPTUAL COMPARISON

ORIGINAL DOCUMENT 1
----------------------------------------------------------------------------------------------------
continue to improve the safety of those models, paving the way for more responsible development of LLMs.
We also share novel observations we made during the development ofLlama 2 and Llama 2-Chat, such as
the emergence of tool usage and temporal organization of knowledge.
3

ORIGINAL DOCUMENT 2
----------------------------------------------------------------------------------------------------
the community to advance AI alignment research.
In this work, we develop and release Llama 2, a family of pretrained and fine-tuned LLMs,Llama 2 and
Llama 2-Chat, at scales up to 70B parameters. On the series of helpfulness and safety benchmarks we tested,
Llama 2-Chat models generally perform better than existing open-source models. They also appear to
be on par with some of the closed-source models, at least on the human evaluations we pe

### Learning: CREATE REUSABLE FUNCTION

**What you'll learn:** Trim retrieved chunks to the spans that answer the query.

**What this cell does:** Defines helper logic for: CREATE REUSABLE FUNCTION.

**Watch for:** Compression is lossy — check that dates/caveats survive.



In [ ]:

# ============================================================
# 13. CREATE REUSABLE FUNCTION
# ============================================================

def contextual_compression_search(
    query: str
):

    documents = (
        compression_retriever.invoke(
            query
        )
    )

    return documents


### Learning: TEST REUSABLE FUNCTION

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `TEST REUSABLE FUNCTION` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:

# ============================================================
# 14. TEST REUSABLE FUNCTION
# ============================================================

query = (
    "What role did human feedback play "
    "in training Llama 2-Chat?"
)


results = contextual_compression_search(
    query
)


print(
    "\n\nREUSABLE CONTEXTUAL COMPRESSION SEARCH"
)

print(
    "=" * 100
)


for i, document in enumerate(
    results,
    start=1
):

    print(
        f"\nRESULT {i}"
    )

    print(
        "Page:",
        document.metadata.get(
            "paper_page"
        )
    )

    print(
        "Section:",
        document.metadata.get(
            "section"
        )
    )

    print(
        "\nRelevant extracted content:"
    )

    print(
        document.page_content
    )

    print(
        "\n" + "-" * 100
    )



REUSABLE CONTEXTUAL COMPRESSION SEARCH

RESULT 1
Page: 5
Section: pretraining

Relevant extracted content:
Figure 4: Training ofLlama 2-Chat: This process begins with thepretraining of Llama 2 using publicly available online sources. Following this, we create an initial version ofLlama 2-Chatthrough the application of supervised fine-tuning. Subsequently, the model is iteratively refined using Reinforcement Learning with Human Feedback(RLHF) methodologies, specifically through rejection sampling and Proximal Policy Optimization (PPO). Throughout the RLHF stage, the accumulation ofiterative reward modeling datain parallel with model enhancements is crucial to ensure the reward models remain within distribution.

----------------------------------------------------------------------------------------------------

RESULT 2
Page: 18
Section: fine_tuning

Relevant extracted content:
3.4.2 Human Evaluation
Human evaluation is often considered the gold standard for judging models for natura

### Learning: FINAL CONCEPTUAL FLOW

**What you'll learn:** Trim retrieved chunks to the spans that answer the query.

**What this cell does:** Runs `FINAL CONCEPTUAL FLOW` and prints intermediate results you can inspect.

**Watch for:** Compression is lossy — check that dates/caveats survive.



In [ ]:
# ============================================================
# 15. FINAL CONCEPTUAL FLOW
# ============================================================

"""
CONTEXTUAL COMPRESSION RETRIEVAL

User Query
      ↓
Base Retriever
      ↓
Retrieve Top-K Documents
      ↓
LLMChainExtractor
      ↓
Query + Document 1
      ↓
Extract relevant content

Query + Document 2
      ↓
Extract relevant content

Query + Document 3
      ↓
Extract relevant content

Query + Document 4
      ↓
Extract relevant content
      ↓
Remove irrelevant information
      ↓
Compressed Documents
      ↓
LLM / RAG Pipeline
"""


print(
    "\nContextual Compression Retriever "
    "practical completed successfully."
)


Contextual Compression Retriever practical completed successfully.


### Learning: CREATE LLM

**What you'll learn:** Authenticate so embedding and chat calls can run.

**What this cell does:** Runs `CREATE LLM` and prints intermediate results you can inspect.

**Watch for:** Never hardcode secrets in shared notebooks.



In [ ]:
# ============================================================
# 4. CREATE LLM
# ============================================================

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

print("LLM created successfully.")